# Hard-Label Adversarial Attacks on Quantum Classifiers

**A Shot-Budget-Aware, Born-Rule-Calibrated Decision-Based Attack for Variational Quantum Classifiers**

Self-contained companion notebook. It writes the full `hlq` package to disk, runs the
sanity gate (T1-T6), executes the experiments for RQ1-RQ5 plus the ablations, and
renders the figures. Everything is reproducible from the seeds in the configs.

**The idea.** Every published adversarial attack on quantum classifiers is white-box and
gradient-based. A deployed quantum classifier returns a *measured class label* - and that
label is a Bernoulli sample whose flip probability is set by the Born-rule margin
`|<M>|` and the shot budget `S`. So the oracle is natively stochastic, and decision-based
attacks are known to break on stochastic oracles. This notebook builds a
**Born-rule-calibrated, shot-budget-aware** hard-label attack and studies its query/shot
economics.

**Runtime notes**
* No GPU needed. These circuits (<=12 qubits) are tiny; the cost is the *number* of
  circuit evaluations, which is CPU- and Python-bound. A Kaggle **CPU** session is a fine
  target - a GPU will not speed this up.
* **Enable Internet** (Notebook settings -> Internet on) so the real MNIST /
  Fashion-MNIST data can download. Alternatively attach the MNIST dataset via *Add Data*;
  the loader will find it.
* Start with `PRESET = "smoke"` to verify the pipeline end-to-end (minutes), then move to
  `"medium"` or `"full"`. `"full"` is the heavier-statistics scope (250 attacked images
  per cell, 8 seeds) and is a multi-hour job - run it in stages with the `RQS` list below.


## 1. Environment

In [ ]:
# PennyLane is the only dependency Kaggle usually lacks.
import importlib, subprocess, sys
for pkg, mod in [("pennylane", "pennylane"), ("pennylane-lightning", "pennylane_lightning")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import pennylane as qml, torch, numpy, scipy, sklearn, matplotlib
print("pennylane   ", qml.__version__)
print("torch       ", torch.__version__)
print("numpy/scipy ", numpy.__version__, scipy.__version__)
print("devices     ", [d for d in qml.plugin_devices] if hasattr(qml, "plugin_devices") else "-")


In [ ]:
import os
os.makedirs("hlq/attacks", exist_ok=True)
os.makedirs("hlq/defenses", exist_ok=True)
os.makedirs("experiments", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)
print("workspace ready")


## 2. The `hlq` package

Every module is written verbatim from the repository sources, so the notebook and the repo cannot diverge.

In [ ]:
%%writefile hlq/__init__.py
"""hlq -- Hard-Label Quantum Attacks.

A shot-budget-aware, Born-rule-calibrated decision-based (hard-label) adversarial
attack framework for variational quantum classifiers (VQCs), plus baselines,
defenses, concentration diagnostics, and research-grade analysis.

Reference: hard_label_quantum_attacks_research_plan.md (Quantum Machine Intelligence).

Design principles
-----------------
* Double-step validation: every numeric component is checked (1) against a
  closed-form / analytic limit and (2) against an independent Monte-Carlo /
  empirical simulation before it is trusted (see hlq.oracle and experiments T1-T6).
* Single source of truth: shared attack machinery lives in ``hlq.attacks.base``;
  concrete attacks override only *policy* hooks (shot budgeting, probe splitting,
  side-of-boundary decision) -- no duplicated boundary-walk code.
* Analytic Born-rule oracle: the exact decision value ``f = <M>`` is computed once
  per query point; the stochastic S-shot label is an *exact* Binomial draw with
  ``p(+1) = (1 + f) / 2`` (diagonal observable), which reproduces real shot-based
  measurement without materializing S samples.
"""

__version__ = "1.0.0"


In [ ]:
%%writefile hlq/seeds.py
"""Reproducibility utilities (the standard ``set_seed`` block, per plan Sec. 8)."""
from __future__ import annotations

import os
import random

import numpy as np


def set_seed(seed: int) -> np.random.Generator:
    """Seed all RNGs used in the project and return a fresh NumPy Generator.

    Seeding ``random``, ``numpy`` (legacy + default hash), ``torch`` (CPU) and
    ``PYTHONHASHSEED`` makes an experiment cell fully determined by ``seed``.
    The returned :class:`numpy.random.Generator` is the *only* stochastic source
    the attacks/oracle should draw from, so that a run is reproducible from the
    single integer ``seed`` regardless of global-state interference.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.use_deterministic_algorithms(False)  # small circuits; keep speed
    except Exception:  # torch optional at import time
        pass
    return np.random.default_rng(seed)


In [ ]:
%%writefile hlq/config.py
"""Typed configuration objects and experiment presets.

Every experiment cell is fully described by a :class:`ClassifierConfig` (the model
under attack), an :class:`AttackConfig` (the adversary), and a
:class:`DefenseConfig` (optional inference-time defense). These are JSON-round-
trippable so ``config.json`` can be dumped at the start of every run (plan Sec. 8).
"""
from __future__ import annotations

import dataclasses
from dataclasses import dataclass, field, asdict
from typing import Optional


# --------------------------------------------------------------------------- #
# Vocabulary (kept as plain strings so configs serialize cleanly to JSON)
# --------------------------------------------------------------------------- #
ENCODINGS = ("angle", "amplitude", "reuploading")
OBSERVABLES = ("local_z", "global_z")          # Z_0  vs  Z^{otimes n}
DATASETS = ("mnist_3v5", "mnist_0v1", "fashion_mnist", "synthetic")
DEFENSES = ("none", "depolarizing", "randomized_encoding")
ATTACKS = (
    "calibrated_hsja",   # ours (M1 + M2)
    "fixed_hsja",        # naive port
    "popskipjump",       # constant-flip baseline
    "pgd_whitebox",      # upper-bound reference
    "momentum",          # momentum-based quantum attack (QMI precedent [6])
    "transfer",          # classical-surrogate transfer
    "classical_hsja",    # HSJA on a matched classical NN (anchor)
)


@dataclass
class ClassifierConfig:
    """Specification of the variational quantum classifier under attack."""

    n_qubits: int = 8
    n_layers: int = 5
    encoding: str = "angle"
    observable: str = "local_z"
    dataset: str = "mnist_3v5"
    reupload_layers: int = 3          # only used by the re-uploading encoding
    epochs: int = 40
    # Large batches are near-free: PennyLane broadcasts the batch through one
    # statevector call, so wall-clock tracks epochs, not steps (~6 s/epoch on the full
    # 11.5k-sample MNIST 3-vs-5 pair at n=8, vs ~10x slower at batch 32).
    batch_size: int = 256
    lr: float = 0.05
    train_size: int = 0               # 0 => use the full available binary-pair train set
    test_size: int = 0                # 0 => use the full available test set
    seed: int = 0

    def __post_init__(self) -> None:
        assert self.encoding in ENCODINGS, self.encoding
        assert self.observable in OBSERVABLES, self.observable
        assert self.dataset in DATASETS, self.dataset

    @property
    def n_features(self) -> int:
        """Input dimensionality fed to the encoder."""
        if self.encoding == "amplitude":
            return 2 ** self.n_qubits
        return self.n_qubits

    def key(self) -> str:
        """Stable identifier used to cache trained models / results."""
        return (
            f"{self.dataset}_n{self.n_qubits}_L{self.n_layers}"
            f"_{self.encoding}_{self.observable}_s{self.seed}"
        )

    to_dict = asdict


@dataclass
class DefenseConfig:
    name: str = "none"
    depolarizing_p: float = 0.05          # per-qubit depolarizing strength
    randomized_strength: float = 0.30     # std of the random-encoder rotations (rad)
    randomized_per_query: bool = True     # resample the random encoder each query

    def __post_init__(self) -> None:
        assert self.name in DEFENSES, self.name

    to_dict = asdict


@dataclass
class AttackConfig:
    """Decision-based attack hyper-parameters (HSJA family)."""

    name: str = "calibrated_hsja"
    iterations: int = 30                  # boundary-walk steps
    total_budget: int = 200_000           # T = total measurement shots for the attack
    delta_decision: float = 0.05          # target per-decision error for M1 (calibrated)
    fixed_shots: int = 100                # S for fixed-shot variants / probe default
    grad_budget_frac: float = 0.5         # fraction of per-iter budget for normal est.
    num_probes: int = 100                 # B (fixed variants / upper cap)
    probe_shots: int = 100                # S_probe (fixed variants)
    init_eval_shots: int = 200            # shots for the (rare) initialisation queries
    bin_search_tol: float = 1e-4          # theta-tolerance of geometric binary search
    norm: str = "l2"                      # perturbation norm ("l2" only for now)
    # PopSkipJump: the CONSTANT flip rate it assumes (oracle-agnostic -- it does not
    # know the Born-rule margin structure, so it applies one repeat count everywhere).
    psj_assumed_p0: float = 0.30
    # PGD reference
    pgd_steps: int = 100
    pgd_step_size: float = 0.02
    pgd_epsilon: float = 0.8
    momentum_mu: float = 1.0              # decay factor for the momentum attack [6]
    seed: int = 0

    def __post_init__(self) -> None:
        assert self.name in ATTACKS, self.name

    to_dict = asdict


@dataclass
class ExperimentConfig:
    """A full experiment cell: model x defense x attack, plus evaluation scope."""

    classifier: ClassifierConfig = field(default_factory=ClassifierConfig)
    defense: DefenseConfig = field(default_factory=DefenseConfig)
    attack: AttackConfig = field(default_factory=AttackConfig)
    n_attack_images: int = 250            # attacked test images per cell (heavier stats)
    seeds: tuple = (0, 1, 2, 3, 4, 5, 6, 7)   # >=5; 8 for heavier statistics

    def to_dict(self) -> dict:
        return {
            "classifier": self.classifier.to_dict(),
            "defense": self.defense.to_dict(),
            "attack": self.attack.to_dict(),
            "n_attack_images": self.n_attack_images,
            "seeds": list(self.seeds),
        }


# --------------------------------------------------------------------------- #
# Presets: fast smoke-scale vs. full "heavier statistics" scope
# --------------------------------------------------------------------------- #
SMOKE = dict(n_attack_images=6, seeds=(0, 1), attack_iterations=8, total_budget=20_000)
FULL = dict(n_attack_images=250, seeds=(0, 1, 2, 3, 4, 5, 6, 7), attack_iterations=30,
            total_budget=200_000)


def dataclass_from_dict(cls, d: dict):
    """Rebuild a dataclass from a plain dict (JSON round-trip helper)."""
    fields = {f.name for f in dataclasses.fields(cls)}
    return cls(**{k: v for k, v in d.items() if k in fields})


In [ ]:
%%writefile hlq/data.py
"""Real datasets (full binary-pair subsets) and the synthetic known-boundary control.

No mock/synthetic stand-ins for real data: MNIST and Fashion-MNIST are downloaded
in full via torchvision and used at their *complete* binary-pair scope.  The only
synthetic dataset is a deliberate control with an *analytically known* decision
boundary, required by sanity test T3 (plan Sec. 5, Sec. 7).

Feature maps
------------
* angle / reuploading : PCA to ``n_qubits`` components (fit on train), then
  min-max scaled to ``[0, pi]`` so features are valid rotation angles.
* amplitude           : each image resized to ``2**n_qubits`` pixels (2-D area
  interpolation, structure preserved), flattened and L2-normalised -> a valid
  quantum state amplitude vector.
"""
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from typing import Optional

import numpy as np

from .config import ClassifierConfig

_CACHE = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "data_cache")

# dataset key -> (torchvision class, (class_neg, class_pos))
_SPEC = {
    "mnist_3v5": ("MNIST", (3, 5)),
    "mnist_0v1": ("MNIST", (0, 1)),
    "fashion_mnist": ("FashionMNIST", (0, 6)),   # T-shirt vs Shirt: a genuinely hard pair
}


@dataclass
class DataBundle:
    """Everything an experiment needs from a dataset, with fitted transforms."""

    X_train: np.ndarray          # (N_tr, n_features) encoder inputs
    y_train: np.ndarray          # (N_tr,) in {-1, +1}
    X_test: np.ndarray           # (N_te, n_features)
    y_test: np.ndarray           # (N_te,) in {-1, +1}
    encoding: str
    n_features: int
    meta: dict                   # provenance + fitted-transform parameters


# --------------------------------------------------------------------------- #
# Raw image loading (full data)
# --------------------------------------------------------------------------- #
def _load_raw_images(dataset: str):
    """Return (Xtr_img, ytr, Xte_img, yte) with images in [0,1], FULL real dataset.

    Tries several providers so the same code runs locally and on Kaggle (whose free
    tier may have Internet disabled): torchvision -> keras -> a mounted /kaggle/input
    copy.  Real data only -- there is no synthetic fallback for MNIST/Fashion-MNIST.
    """
    tv_name, _ = _SPEC[dataset]
    os.makedirs(_CACHE, exist_ok=True)
    errors = []

    try:                                   # 1. torchvision (cached to data_cache/)
        import torchvision

        cls = getattr(torchvision.datasets, tv_name)
        tr = cls(_CACHE, train=True, download=True)
        te = cls(_CACHE, train=False, download=True)
        return (tr.data.numpy().astype(np.float64) / 255.0, np.asarray(tr.targets),
                te.data.numpy().astype(np.float64) / 255.0, np.asarray(te.targets))
    except Exception as e:                 # pragma: no cover - provider fallback
        errors.append(f"torchvision: {e!r}")

    try:                                   # 2. keras bundled loaders
        from tensorflow import keras

        loader = (keras.datasets.mnist if tv_name == "MNIST"
                  else keras.datasets.fashion_mnist)
        (Xtr, ytr), (Xte, yte) = loader.load_data()
        return (Xtr.astype(np.float64) / 255.0, ytr,
                Xte.astype(np.float64) / 255.0, yte)
    except Exception as e:                 # pragma: no cover
        errors.append(f"keras: {e!r}")

    # 3. a dataset mounted into a Kaggle notebook (Add Data -> MNIST / Fashion-MNIST)
    for base in ("/kaggle/input", os.path.join(_CACHE, "kaggle")):
        hit = _try_kaggle_idx(base, tv_name)
        if hit is not None:
            return hit
    raise RuntimeError(
        f"Could not load real {dataset}. Enable Internet in the notebook settings, or "
        f"attach the dataset. Attempts: " + " | ".join(errors))


def _try_kaggle_idx(base: str, tv_name: str):
    """Look for raw idx-ubyte files in a mounted directory tree."""
    if not os.path.isdir(base):
        return None
    want = {"train_x": "train-images", "train_y": "train-labels",
            "test_x": "t10k-images", "test_y": "t10k-labels"}
    found = {}
    for root, _dirs, files in os.walk(base):
        for f in files:
            for k, pat in want.items():
                if pat in f and "idx" in f and k not in found:
                    found[k] = os.path.join(root, f)
    if len(found) < 4:
        return None
    Xtr = _read_idx(found["train_x"]).astype(np.float64) / 255.0
    Xte = _read_idx(found["test_x"]).astype(np.float64) / 255.0
    return Xtr, _read_idx(found["train_y"]), Xte, _read_idx(found["test_y"])


def _read_idx(path: str) -> np.ndarray:
    """Minimal IDX (MNIST binary format) reader, transparently gunzipping."""
    import gzip
    import struct

    op = gzip.open if path.endswith(".gz") else open
    with op(path, "rb") as fh:
        magic, ndim = struct.unpack(">HBB", fh.read(4))[1:]
        shape = struct.unpack(">" + "I" * ndim, fh.read(4 * ndim))
        return np.frombuffer(fh.read(), dtype=np.uint8).reshape(shape)


def _binary_filter(X, y, classes):
    neg, pos = classes
    mask = (y == neg) | (y == pos)
    Xb = X[mask]
    yb = np.where(y[mask] == pos, 1, -1).astype(np.int64)
    return Xb, yb


# --------------------------------------------------------------------------- #
# Feature maps
# --------------------------------------------------------------------------- #
def _resize_to_pow2(imgs: np.ndarray, n_qubits: int) -> np.ndarray:
    """Area-resize (N,H,W) images to 2**n pixels, preserving 2-D structure."""
    import torch
    import torch.nn.functional as F

    h = 2 ** ((n_qubits + 1) // 2)
    w = 2 ** (n_qubits // 2)
    t = torch.from_numpy(imgs).unsqueeze(1).float()          # (N,1,H,W)
    t = F.interpolate(t, size=(h, w), mode="area")
    return t.squeeze(1).reshape(imgs.shape[0], h * w).numpy().astype(np.float64)


def _make_features(Xtr_img, Xte_img, cfg: ClassifierConfig):
    """Build (Xtr, Xte, meta) for the requested encoding. Transforms fit on train."""
    n = cfg.n_qubits
    if cfg.encoding == "amplitude":
        Ftr = _resize_to_pow2(Xtr_img, n)
        Fte = _resize_to_pow2(Xte_img, n)
        ntr = np.linalg.norm(Ftr, axis=1, keepdims=True)
        nte = np.linalg.norm(Fte, axis=1, keepdims=True)
        ntr[ntr == 0] = 1.0
        nte[nte == 0] = 1.0
        Ftr, Fte = Ftr / ntr, Fte / nte
        meta = {"map": "resize_l2norm", "dim": 2 ** n}
        return Ftr, Fte, meta

    # angle / reuploading: PCA -> min-max to [0, pi]
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import MinMaxScaler

    flat_tr = Xtr_img.reshape(Xtr_img.shape[0], -1)
    flat_te = Xte_img.reshape(Xte_img.shape[0], -1)
    pca = PCA(n_components=n, random_state=0).fit(flat_tr)
    Ztr, Zte = pca.transform(flat_tr), pca.transform(flat_te)
    scaler = MinMaxScaler((0.0, np.pi)).fit(Ztr)
    Ftr = scaler.transform(Ztr)
    Fte = np.clip(scaler.transform(Zte), 0.0, np.pi)         # keep test angles valid
    meta = {
        "map": "pca_minmax",
        "pca_explained_var": float(pca.explained_variance_ratio_.sum()),
        "n_components": n,
    }
    return Ftr, Fte, meta


def _feature_cache_path(cfg: ClassifierConfig) -> str:
    os.makedirs(_CACHE, exist_ok=True)
    return os.path.join(
        _CACHE,
        f"feat_{cfg.dataset}_n{cfg.n_qubits}_{cfg.encoding}"
        f"_tr{cfg.train_size}_te{cfg.test_size}_s{cfg.seed}.npz",
    )


def load_dataset(cfg: ClassifierConfig, use_cache: bool = True) -> DataBundle:
    """Load a real binary-pair dataset (full scope) as encoder-ready features.

    Features are cached to disk: the PCA/resize transform is deterministic given the
    config, so parallel workers reuse it instead of recomputing it per process.
    """
    if cfg.dataset == "synthetic":
        return _load_synthetic(cfg)

    cache = _feature_cache_path(cfg)
    if use_cache and os.path.exists(cache):
        d = np.load(cache, allow_pickle=True)
        return DataBundle(d["X_train"], d["y_train"], d["X_test"], d["y_test"],
                          cfg.encoding, cfg.n_features, json.loads(str(d["meta"])))

    Xtr_img, ytr_all, Xte_img, yte_all = _load_raw_images(cfg.dataset)
    Xtr_img, ytr = _binary_filter(Xtr_img, ytr_all, _SPEC[cfg.dataset][1])
    Xte_img, yte = _binary_filter(Xte_img, yte_all, _SPEC[cfg.dataset][1])

    # Optional caps (train_size/test_size == 0 => full data)
    rng = np.random.default_rng(cfg.seed)
    if cfg.train_size and cfg.train_size < len(ytr):
        idx = rng.permutation(len(ytr))[: cfg.train_size]
        Xtr_img, ytr = Xtr_img[idx], ytr[idx]
    if cfg.test_size and cfg.test_size < len(yte):
        idx = rng.permutation(len(yte))[: cfg.test_size]
        Xte_img, yte = Xte_img[idx], yte[idx]

    Xtr, Xte, meta = _make_features(Xtr_img, Xte_img, cfg)
    meta.update(dataset=cfg.dataset, classes=list(_SPEC[cfg.dataset][1]),
                n_train=int(len(ytr)), n_test=int(len(yte)))
    if use_cache:
        np.savez_compressed(cache, X_train=Xtr, y_train=ytr, X_test=Xte, y_test=yte,
                            meta=json.dumps(meta))
    return DataBundle(Xtr, ytr, Xte, yte, cfg.encoding, cfg.n_features, meta)


# --------------------------------------------------------------------------- #
# Synthetic control with analytically known boundary (T3)
# --------------------------------------------------------------------------- #
def _load_synthetic(cfg: ClassifierConfig) -> DataBundle:
    """Linearly separable data in feature space with a known margin boundary.

    Feature space matches the angle encoder ([0, pi]); label = sign(w.x + b).
    The exact point-to-boundary distance is |w.x + b| / ||w||, used by T3.
    """
    n = cfg.n_qubits
    rng = np.random.default_rng(cfg.seed)
    w, b = make_linear_boundary(n, seed=cfg.seed)
    n_tr, n_te = 4000, 2000

    def _sample(m):
        X = rng.uniform(0.0, np.pi, size=(m, n))
        s = X @ w + b
        keep = np.abs(s) > 0.15                      # small margin band removed
        X, s = X[keep], s[keep]
        y = np.where(s > 0, 1, -1).astype(np.int64)
        return X, y

    Xtr, ytr = _sample(int(n_tr * 1.4))
    Xte, yte = _sample(int(n_te * 1.4))
    Xtr, ytr = Xtr[:n_tr], ytr[:n_tr]
    Xte, yte = Xte[:n_te], yte[:n_te]
    meta = {"map": "synthetic_linear", "w": w.tolist(), "b": float(b),
            "dataset": "synthetic", "n_train": len(ytr), "n_test": len(yte)}
    return DataBundle(Xtr, ytr, Xte, yte, "angle", n, meta)


def make_linear_boundary(n: int, seed: int = 0):
    """A fixed random hyperplane through the centre of the [0, pi]^n box."""
    rng = np.random.default_rng(seed + 777)
    w = rng.normal(size=n)
    w /= np.linalg.norm(w)
    b = -float(w @ (np.full(n, np.pi / 2)))          # boundary passes box centre
    return w, b


In [ ]:
%%writefile hlq/classifier.py
"""Variational quantum classifier under attack (plan Sec. 4.1).

    y_hat(x) = sign(f_theta(x)),
    f_theta(x) = <0| E(x)^dag W(theta)^dag M W(theta) E(x) |0>,  f in [-1, 1].

One ``_circuit`` definition is shared across three QNodes so the *same* unitary is
used everywhere (single source of truth):

* inference QNode  (lightning.qubit, batched)  -> exact ``f`` for the oracle;
* training QNode   (default.qubit, torch backprop) -> gradient-based fitting;
* the training QNode is reused for white-box input gradients (PGD reference).

VERIFY invariants (asserted): |f| <= 1 for every input; loss decreases over the
first epochs; clean decision values are non-degenerate (checked by callers/T-tests).
"""
from __future__ import annotations

import os
from typing import Optional

import numpy as np
import pennylane as qml

from .config import ClassifierConfig

_MODELS = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "models_cache")


# --------------------------------------------------------------------------- #
# Circuit building blocks (shared by all QNodes)
# --------------------------------------------------------------------------- #
def _observable(cfg: ClassifierConfig):
    """Hermitian measurement M with eigenvalues in {-1,+1} (diagonal in Z basis)."""
    if cfg.observable == "local_z":
        return qml.PauliZ(0)
    op = qml.PauliZ(0)                     # global parity Z^{otimes n} (concentration source)
    for i in range(1, cfg.n_qubits):
        op = op @ qml.PauliZ(i)
    return op


def weight_shape(cfg: ClassifierConfig):
    if cfg.encoding == "reuploading":
        return (cfg.reupload_layers, cfg.n_qubits, 3)
    return qml.StronglyEntanglingLayers.shape(n_layers=cfg.n_layers, n_wires=cfg.n_qubits)


def apply_encoding(x, cfg: ClassifierConfig):
    """Data-encoding unitary E(x) (single application; broadcasts over batch)."""
    wires = range(cfg.n_qubits)
    if cfg.encoding == "amplitude":
        qml.AmplitudeEmbedding(x, wires=wires, normalize=True, pad_with=0.0)
    else:                                         # angle / reuploading base
        qml.AngleEmbedding(x, wires=wires, rotation="Y")


def apply_ansatz(weights, cfg: ClassifierConfig):
    qml.StronglyEntanglingLayers(weights, wires=range(cfg.n_qubits))


def circuit_body(x, weights, cfg: ClassifierConfig):
    """E(x) then W(theta), with re-uploading interleaving encode/ansatz per layer."""
    wires = range(cfg.n_qubits)
    if cfg.encoding == "reuploading":
        for r in range(cfg.reupload_layers):      # data re-uploading (Perez-Salinas)
            qml.AngleEmbedding(x, wires=wires, rotation="Y")
            qml.StronglyEntanglingLayers(weights[r : r + 1], wires=wires)
    else:
        apply_encoding(x, cfg)
        apply_ansatz(weights, cfg)


def _circuit(x, weights, cfg: ClassifierConfig):
    """E(x) then W(theta); returns <M>. Broadcasts over a leading batch dim of x."""
    circuit_body(x, weights, cfg)
    return qml.expval(_observable(cfg))


# --------------------------------------------------------------------------- #
# Classifier
# --------------------------------------------------------------------------- #
class VQC:
    """Trainable VQC with a fast batched exact-inference path for the oracle."""

    def __init__(self, cfg: ClassifierConfig):
        self.cfg = cfg
        self.weights: Optional[np.ndarray] = None
        self._infer_qnode = None
        self._torch_qnode = None

    # -- QNode construction ------------------------------------------------- #
    def _build_infer(self):
        if self._infer_qnode is None:
            try:
                dev = qml.device("lightning.qubit", wires=self.cfg.n_qubits)
            except Exception:
                dev = qml.device("default.qubit", wires=self.cfg.n_qubits)
            cfg = self.cfg
            self._infer_qnode = qml.QNode(lambda x, w: _circuit(x, w, cfg), dev)
        return self._infer_qnode

    def _build_torch(self):
        if self._torch_qnode is None:
            dev = qml.device("default.qubit", wires=self.cfg.n_qubits)
            cfg = self.cfg
            self._torch_qnode = qml.QNode(
                lambda x, w: _circuit(x, w, cfg), dev,
                interface="torch", diff_method="backprop",
            )
        return self._torch_qnode

    # -- Inference ---------------------------------------------------------- #
    def decision_function(self, X: np.ndarray) -> np.ndarray:
        """Exact infinite-shot decision value f_theta(x), batched. No gradients."""
        X = np.atleast_2d(np.asarray(X, dtype=np.float64))
        qn = self._build_infer()
        out = np.asarray(qn(X, self.weights), dtype=np.float64).reshape(-1)
        # VERIFY: observable value must lie in [-1, 1] (numerical guard).
        assert np.all(np.abs(out) <= 1.0 + 1e-6), f"|f|>1: max={np.abs(out).max()}"
        return np.clip(out, -1.0, 1.0)

    def predict(self, X: np.ndarray) -> np.ndarray:
        f = self.decision_function(X)
        return np.where(f >= 0, 1, -1).astype(np.int64)

    def clean_accuracy(self, X, y) -> float:
        return float(np.mean(self.predict(X) == np.asarray(y)))

    def margins(self, X) -> np.ndarray:
        """|f_theta(x)| -- used for the concentration diagnostic (RQ5)."""
        return np.abs(self.decision_function(X))

    # -- Differentiable path (white-box PGD reference) ---------------------- #
    def decision_function_torch(self, X_torch):
        import torch

        qn = self._build_torch()
        w = torch.as_tensor(self.weights, dtype=torch.float64)
        out = qn(X_torch, w)
        return out.reshape(-1)

    # -- Training ----------------------------------------------------------- #
    def fit(self, X, y, X_test=None, y_test=None, *, verbose=False, use_cache=True) -> dict:
        """Train theta by Adam on binary cross-entropy of p=(1+f)/2.

        Always returns the same info dict and handles its own weight cache, so a cache
        hit and a fresh fit are indistinguishable to callers. Models are keyed by the
        full config, so one trained VQC is reused across every attack, defense and RQ.
        """
        import torch

        os.makedirs(_MODELS, exist_ok=True)
        cache = os.path.join(_MODELS, self.cfg.key() + ".npz")
        if use_cache and os.path.exists(cache):
            d = np.load(cache)
            self.weights = d["weights"]
            return {"train_acc": float(d["train_acc"]), "test_acc": float(d["test_acc"]),
                    "loss_curve": d["loss_curve"].tolist(), "cached": True}

        torch.manual_seed(self.cfg.seed)
        shape = weight_shape(self.cfg)
        w = torch.nn.Parameter(0.1 * torch.randn(*shape, dtype=torch.float64))
        qn = self._build_torch()
        Xt = torch.as_tensor(np.asarray(X), dtype=torch.float64)
        yt = torch.as_tensor((np.asarray(y) + 1) // 2, dtype=torch.float64)   # {0,1}
        opt = torch.optim.Adam([w], lr=self.cfg.lr)

        n = len(yt)
        bs = self.cfg.batch_size
        loss_curve = []
        for epoch in range(self.cfg.epochs):
            perm = torch.randperm(n)
            ep_loss = 0.0
            for i in range(0, n, bs):
                idx = perm[i : i + bs]
                opt.zero_grad()
                f = qn(Xt[idx], w).reshape(-1)
                p = torch.clamp((1.0 + f) / 2.0, 1e-6, 1 - 1e-6)
                loss = torch.nn.functional.binary_cross_entropy(p, yt[idx])
                loss.backward()
                opt.step()
                ep_loss += float(loss) * len(idx)
            loss_curve.append(ep_loss / n)
            if verbose and (epoch % 5 == 0 or epoch == self.cfg.epochs - 1):
                self.weights = w.detach().numpy()
                print(f"  epoch {epoch:3d} loss={loss_curve[-1]:.4f} "
                      f"acc={self.clean_accuracy(X, y):.3f}")

        self.weights = w.detach().numpy()
        # VERIFY: the loss must fall over the first epochs (guards a degenerate fit)
        if len(loss_curve) >= 10:
            assert loss_curve[9] <= loss_curve[0] + 1e-6, "loss did not decrease early"

        train_acc = self.clean_accuracy(X, y)
        test_acc = (self.clean_accuracy(X_test, y_test)
                    if X_test is not None else float("nan"))
        np.savez(cache, weights=self.weights, train_acc=train_acc, test_acc=test_acc,
                 loss_curve=np.asarray(loss_curve))
        return {"train_acc": float(train_acc), "test_acc": float(test_acc),
                "loss_curve": loss_curve, "cached": False}


def train_or_load(cfg: ClassifierConfig, bundle, verbose=False):
    """Train (or load cached) a VQC on a DataBundle. Returns (vqc, info)."""
    vqc = VQC(cfg)
    info = vqc.fit(bundle.X_train, bundle.y_train, bundle.X_test, bundle.y_test,
                   verbose=verbose)
    return vqc, info


In [ ]:
%%writefile hlq/classical.py
"""Parameter-matched classical NN -- the classical anchor (plan Sec. 4.4).

HSJA on this model calibrates whether the quantum attack's query cost is "normal":
the same boundary-walk skeleton runs against a *deterministic* oracle (S=None), so
any extra cost the VQC imposes is attributable to its stochastic Born-rule oracle
rather than to the attack.  Hidden width is chosen so the parameter count matches the
VQC's ansatz (``L * n * 3`` for StronglyEntanglingLayers).
"""
from __future__ import annotations

import numpy as np

from .config import ClassifierConfig


def matched_hidden_width(cfg: ClassifierConfig) -> int:
    """Hidden width h giving ~ the VQC's parameter count for input dim d."""
    d = cfg.n_features
    target = (cfg.reupload_layers if cfg.encoding == "reuploading" else cfg.n_layers) \
        * cfg.n_qubits * 3
    h = int(round(target / max(d + 2, 1)))
    return max(2, h)


class MatchedClassicalNN:
    """Tanh-output MLP exposing the same interface as :class:`hlq.classifier.VQC`."""

    def __init__(self, cfg: ClassifierConfig):
        self.cfg = cfg
        self.hidden = matched_hidden_width(cfg)
        self.net = None

    def _build(self):
        import torch

        torch.manual_seed(self.cfg.seed)
        self.net = torch.nn.Sequential(
            torch.nn.Linear(self.cfg.n_features, self.hidden), torch.nn.Tanh(),
            torch.nn.Linear(self.hidden, 1), torch.nn.Tanh(),      # output in [-1,1]
        ).double()
        return self.net

    def n_parameters(self) -> int:
        if self.net is None:
            self._build()
        return int(sum(p.numel() for p in self.net.parameters()))

    def fit(self, X, y, epochs=200, lr=0.01):
        import torch

        net = self._build()
        Xt = torch.tensor(np.asarray(X), dtype=torch.float64)
        yt = torch.tensor((np.asarray(y) + 1) / 2.0, dtype=torch.float64).reshape(-1, 1)
        opt = torch.optim.Adam(net.parameters(), lr=lr)
        for _ in range(epochs):
            opt.zero_grad()
            f = net(Xt)
            p = torch.clamp((1.0 + f) / 2.0, 1e-6, 1 - 1e-6)
            loss = torch.nn.functional.binary_cross_entropy(p, yt)
            loss.backward()
            opt.step()
        return {"train_acc": self.clean_accuracy(X, y), "n_parameters": self.n_parameters(),
                "hidden": self.hidden}

    def decision_function(self, X):
        import torch

        if self.net is None:
            self._build()
        X = np.atleast_2d(np.asarray(X, dtype=np.float64))
        with torch.no_grad():
            out = self.net(torch.tensor(X, dtype=torch.float64)).numpy().reshape(-1)
        return np.clip(out, -1.0, 1.0)

    def predict(self, X):
        return np.where(self.decision_function(X) >= 0, 1, -1).astype(np.int64)

    def clean_accuracy(self, X, y):
        return float(np.mean(self.predict(X) == np.asarray(y)))

    def margins(self, X):
        return np.abs(self.decision_function(X))


In [ ]:
%%writefile hlq/oracle.py
"""The stochastic label oracle -- the quantum-specific crux (plan Sec. 4.2).

A quantum classifier queried with a finite shot budget ``S`` is a *natively
stochastic* oracle.  For a diagonal observable ``M`` with eigenvalues in {-1,+1},
each shot is an independent Bernoulli with ``p(+1) = (1 + f)/2`` where
``f = <M>`` is the exact decision value.  Hence the S-shot empirical mean and its
sign are governed *exactly* by a Binomial(S, (1+f)/2) draw -- this reproduces real
shot-based measurement without materialising S samples, and is what every attack
queries.

Two models live here, and the paper's whole premise is the gap between them:

* ``sample_label`` -- the EXACT Binomial oracle (ground truth the attacks face);
* ``p_flip`` -- the closed-form CLT/Gaussian model the *calibrated* attacker uses
  to choose its shot budget ``S(delta, m_hat)``.

``validate_p_flip`` performs the double-step check (T1): closed-form vs an
independent Monte-Carlo estimate of the flip rate, plus the three analytic limits.
"""
from __future__ import annotations

from typing import Optional

import numpy as np
from scipy.stats import norm


# --------------------------------------------------------------------------- #
# Closed-form Born-rule flip model (the attacker's calibration knowledge)
# --------------------------------------------------------------------------- #
def p_flip(f: np.ndarray, S: int) -> np.ndarray:
    """P(sign(f_hat) != sign(f)) under the CLT model:  Phi( -|f| sqrt(S/(1-f^2)) ).

    Analytic limits (VERIFY): S->inf => 0 ; |f|->0 => 1/2 ; monotone down in S,|f|.
    """
    f = np.asarray(f, dtype=np.float64)
    m = np.abs(f)
    var = np.clip(1.0 - f * f, 1e-15, None)          # Var[f_hat] = (1-f^2)/S
    z = m * np.sqrt(S / var)
    return norm.cdf(-z)


def shots_for_delta(f_hat: float, delta: float, s_min: int = 1,
                    s_max: int = 100_000) -> int:
    """Smallest S guaranteeing p_flip <= delta at estimated decision value f_hat.

    Invert the CLT model:  S = ceil( (1 - f_hat^2) * (Phi^{-1}(delta) / |f_hat|)^2 ).
    """
    m = abs(float(f_hat))
    if m < 1e-9:                                     # on the boundary: never reliable
        return s_max
    z = norm.ppf(delta)                              # < 0 for delta < 0.5
    s = int(np.ceil((1.0 - f_hat * f_hat) * (z / m) ** 2))
    return int(np.clip(s, s_min, s_max))


# --------------------------------------------------------------------------- #
# Exact stochastic oracle
# --------------------------------------------------------------------------- #
def sample_label(f: np.ndarray, S: Optional[int], rng: np.random.Generator) -> np.ndarray:
    """Exact S-shot label(s) from decision value(s) f. S=None => deterministic sign.

    k ~ Binomial(S, (1+f)/2);  f_hat = 2k/S - 1;  label = sign(f_hat), ties broken
    by a fair coin (matches an undefined sign on the measured boundary).
    """
    f = np.asarray(f, dtype=np.float64)
    scalar = f.ndim == 0
    f = np.atleast_1d(f)
    if S is None or S == np.inf:
        lab = np.where(f >= 0, 1, -1)
    else:
        p = np.clip((1.0 + f) / 2.0, 0.0, 1.0)
        k = rng.binomial(int(S), p)
        fhat = 2.0 * k / S - 1.0
        lab = np.sign(fhat).astype(np.int64)
        ties = lab == 0
        if ties.any():
            lab[ties] = rng.choice([-1, 1], size=int(ties.sum()))
    lab = lab.astype(np.int64)
    return lab[0] if scalar else lab


class BudgetExhausted(Exception):
    """Raised when a query would exceed the oracle's total measurement budget T."""


class StochasticOracle:
    """Hard-label oracle over a trained classifier, with query/shot accounting.

    The attacks see ONLY ``label`` / ``label_batch`` (top-1 hard labels) and the
    original label; they never receive ``f``.  ``estimate_f`` is available for the
    calibrated attack's shot controller (it costs shots, which are accounted).

    A finite ``budget`` (total shots ``T``) makes every attack stop at *exactly* the
    same measurement budget: a query that would exceed ``T`` raises
    :class:`BudgetExhausted`, which the attack skeleton catches and returns best-so-far.
    This is what makes the fixed-``T`` comparisons (RQ2/RQ3) fair.
    """

    def __init__(self, classifier, rng: np.random.Generator, *, stochastic=False,
                 budget: float = float("inf"), cache_f: bool = True, cache_max=50_000):
        self.clf = classifier
        self.rng = rng
        self.stochastic = stochastic          # True for randomized-encoding defense
        self.budget = budget
        self.n_queries = 0
        self.n_shots = 0
        self.n_circuit_evals = 0              # wall-clock cost (cache misses only)
        # f(x) is a deterministic pure function, so repeated queries at the SAME point
        # (the calibrated attack's shot escalation, PopSkipJump's repeats) need only one
        # simulation.  Shots are still independent Binomial draws from that same f, so
        # the statistics are unchanged -- this is purely a compute optimisation.
        # Disabled for per-query-stochastic defenses, whose f genuinely varies per call.
        self._fcache = {} if (cache_f and not stochastic) else None
        self._cache_max = cache_max

    def reset_counters(self):
        self.n_queries = 0
        self.n_shots = 0
        self.n_circuit_evals = 0

    def _charge(self, cost: int):
        if self.n_shots + cost > self.budget:
            raise BudgetExhausted
        self.n_shots += int(cost)

    def _f_single(self, x) -> float:
        if self._fcache is None:
            self.n_circuit_evals += 1
            return float(self.clf.decision_function(np.atleast_2d(x))[0])
        key = np.asarray(x, dtype=np.float64).tobytes()
        v = self._fcache.get(key)
        if v is None:
            self.n_circuit_evals += 1
            v = float(self.clf.decision_function(np.atleast_2d(x))[0])
            if len(self._fcache) >= self._cache_max:
                self._fcache.clear()
            self._fcache[key] = v
        return v

    # -- exact decision value (no measurement noise; not charged as a query) -- #
    def f_exact(self, X):
        return self.clf.decision_function(X)

    # -- hard-label queries (charged) -------------------------------------- #
    def label(self, x, S: Optional[int]) -> int:
        self._charge(int(S) if S not in (None, np.inf) else 0)
        self.n_queries += 1
        return int(sample_label(self._f_single(x), S, self.rng))

    def label_batch(self, X, S: Optional[int]) -> np.ndarray:
        X = np.atleast_2d(X)
        self._charge(len(X) * (int(S) if S not in (None, np.inf) else 0))
        self.n_queries += len(X)
        self.n_circuit_evals += len(X)                  # batched, but still len(X) evals
        f = self.clf.decision_function(X)
        return sample_label(f, S, self.rng)

    def estimate_f(self, x, S: int) -> float:
        """Empirical f_hat over S shots (single Binomial draw). Charged as 1 query."""
        self._charge(int(S))
        self.n_queries += 1
        p = np.clip((1.0 + self._f_single(x)) / 2.0, 0.0, 1.0)
        k = self.rng.binomial(int(S), p)
        return 2.0 * k / S - 1.0

    def estimate_f_batch(self, X, S: int) -> np.ndarray:
        """Empirical f_hat over S shots for a batch (one circuit call). Charged per point."""
        X = np.atleast_2d(X)
        self._charge(len(X) * int(S))
        self.n_queries += len(X)
        self.n_circuit_evals += len(X)
        f = self.clf.decision_function(X)
        p = np.clip((1.0 + f) / 2.0, 0.0, 1.0)
        k = self.rng.binomial(int(S), p)
        return 2.0 * k / S - 1.0


# --------------------------------------------------------------------------- #
# Double-step validation of the flip model  (sanity test T1)
# --------------------------------------------------------------------------- #
def validate_p_flip(margins=None, shots=None, trials=20000, seed=0) -> dict:
    """Closed-form p_flip vs an independent Monte-Carlo flip rate over an (m,S) grid.

    Returns a JSON-serialisable record with the grid, both surfaces, the max
    absolute discrepancy, and explicit checks of the three analytic limits.
    """
    rng = np.random.default_rng(seed)
    if margins is None:
        margins = np.linspace(0.02, 0.98, 25)
    if shots is None:
        shots = np.array([10, 30, 100, 300, 1000, 3000, 10000])
    margins = np.asarray(margins, dtype=float)
    shots = np.asarray(shots, dtype=int)

    closed = np.zeros((len(margins), len(shots)))
    mc = np.zeros_like(closed)
    for i, m in enumerate(margins):
        f = m                                        # true label +1 (f>0)
        for j, S in enumerate(shots):
            closed[i, j] = float(p_flip(f, int(S)))
            labs = sample_label(np.full(trials, f), int(S), rng)
            mc[i, j] = float(np.mean(labs != 1))     # empirical flip rate vs true +1

    mc_se = np.sqrt(np.clip(mc * (1 - mc), 1e-12, None) / trials)
    within = np.abs(closed - mc) <= (3 * mc_se + 5e-3)

    # analytic limits
    lim_inf = float(p_flip(0.5, 10_000_000))                       # S->inf
    lim_boundary = float(p_flip(1e-6, 1000))                       # m->0
    mono_S = bool(np.all(np.diff([p_flip(0.3, s) for s in [10, 100, 1000, 10000]]) <= 1e-9))
    mono_m = bool(np.all(np.diff([p_flip(mm, 200) for mm in [0.1, 0.3, 0.5, 0.7, 0.9]]) <= 1e-9))

    return {
        "test": "T1_p_flip",
        "margins": margins.tolist(),
        "shots": shots.tolist(),
        "p_flip_closed": closed.tolist(),
        "p_flip_montecarlo": mc.tolist(),
        "max_abs_discrepancy": float(np.abs(closed - mc).max()),
        "fraction_within_3se": float(np.mean(within)),
        "limit_S_to_inf": lim_inf,
        "limit_margin_to_0": lim_boundary,
        "monotone_decreasing_in_S": mono_S,
        "monotone_decreasing_in_margin": mono_m,
        "trials": int(trials),
        "passed": bool(lim_inf < 1e-3 and abs(lim_boundary - 0.5) < 0.05
                       and mono_S and mono_m and np.mean(within) > 0.9),
    }


In [ ]:
%%writefile hlq/budget.py
"""Query/shot economics of the boundary-normal estimate (RQ2, plan Sec. 4.3-M2).

At a boundary point the attacker estimates the normal by averaging the +-1 labels of
``B`` random probes, each measured with ``S_probe`` shots.  A probe's label is the
true side with probability ``1 - p_flip(m, S_probe)``, so its signed contribution is
attenuated by ``(1 - 2 p_flip)``.  Averaging ``B = T_grad / S_probe`` such probes gives

    SNR^2(direction)  ∝  B (1 - 2 p_flip)^2  =  T_grad * (1 - 2 p_flip(m, S))^2 / S.

For fixed total gradient budget ``T_grad`` the optimum is therefore the ``S`` that
maximises ``g(S) = (1 - 2 p_flip(m, S))^2 / S`` -- an interior optimum (RQ2/H2):
too few shots -> labels near the boundary are coin-flips (numerator -> 0); too many
shots -> too few probes (denominator grows).  This module is the single source of
that objective, used both by the calibrated attack (M2) and by the RQ2 theory curve.
"""
from __future__ import annotations

import numpy as np

from .oracle import p_flip


def grad_snr_per_budget(margin: float, S) -> np.ndarray:
    """The objective g(S) = (1 - 2 p_flip(m, S))^2 / S (SNR^2 per unit T_grad)."""
    S = np.asarray(S, dtype=float)
    p = p_flip(float(margin), S.astype(int) if S.ndim else int(S))
    return (1.0 - 2.0 * p) ** 2 / S


def optimal_probe_split(T_grad: int, margin: float, *, min_probes: int = 8,
                        max_probes: int = 256, shots_grid=None) -> tuple:
    """Choose (B, S_probe) maximising the gradient SNR at fixed budget T_grad.

    ``max_probes`` bounds ``B`` (equivalently floors ``S_probe`` at
    ``T_grad / max_probes``).  This is a *practical* constraint, not a statistical
    one: the unconstrained objective is flat as ``S -> 1`` (see module docstring), so
    the optimiser would otherwise request tens of thousands of single-shot probes per
    iteration.  Each probe is still a circuit evaluation, and a real deployed API also
    rate-limits queries, so B is capped.  The RQ2 sweep reports the *unconstrained*
    curve so the plateau/boundary optimum is visible regardless of this cap.

    Returns ``(B, S_probe, info)`` with the swept grid and curve for plotting/auditing.
    """
    T_grad = int(max(T_grad, min_probes))
    s_hi = max(1, T_grad // min_probes)                    # B >= min_probes
    s_lo = max(1, T_grad // max(1, max_probes))            # B <= max_probes
    if shots_grid is None:
        shots_grid = np.unique(np.round(np.geomspace(1, max(2, s_hi), 40)).astype(int))
    shots_grid = np.asarray(shots_grid, dtype=int)
    g = np.array([grad_snr_per_budget(margin, int(S)) for S in shots_grid])
    feasible = (shots_grid <= s_hi) & (shots_grid >= s_lo)
    g_feas = np.where(feasible, g, -np.inf)
    j = int(np.argmax(g_feas))
    S_probe = int(shots_grid[j])
    B = int(np.clip(T_grad // S_probe, min_probes, max_probes))
    info = {"shots_grid": shots_grid.tolist(),
            "snr": [float(v) for v in g],                  # unconstrained curve
            "feasible": [bool(v) for v in feasible],
            "margin": float(margin), "T_grad": int(T_grad)}
    return B, S_probe, info


def predicted_optimal_shots(T_grad: int, margin: float, **kw) -> int:
    """The theory-predicted S* (used to overlay on the empirical RQ2 curve)."""
    return optimal_probe_split(T_grad, margin, **kw)[1]


In [ ]:
%%writefile hlq/metrics.py
"""Evaluation metrics with ground-truth verification (plan Sec. 6).

A decision-based attack only ever sees *noisy* labels, so it can believe it has
crossed the boundary when it has not.  Every returned adversarial is therefore
re-checked against the exact (infinite-shot) model: a run counts as a success only
if the returned point is *truly* adversarial.  This is what exposes the failure of
the naive fixed-shot port and rewards the calibrated attack (RQ1/RQ3).
"""
from __future__ import annotations

from typing import Optional

import numpy as np
from scipy import stats


# --------------------------------------------------------------------------- #
# Ground-truth verification
# --------------------------------------------------------------------------- #
def verify(vqc, x0, x_adv, y0) -> dict:
    """Check a returned adversarial against the exact model.

    Returns verified success and the verified L2 perturbation (inf if the point is
    not truly adversarial).  ``exact_margin`` is the signed decision value at x_adv.
    """
    x0 = np.asarray(x0, float)
    x_adv = np.asarray(x_adv, float)
    f = float(vqc.decision_function(x_adv.reshape(1, -1))[0])
    truly_adv = (1 if f >= 0 else -1) != int(y0)
    pert = float(np.linalg.norm(x_adv - x0)) if truly_adv else float("inf")
    return {"verified_success": bool(truly_adv), "verified_perturbation": pert,
            "exact_margin_at_adv": abs(f)}


def shots_to_threshold(result_record: dict, verified: bool, eps: float) -> float:
    """Shots spent to first reach believed perturbation <= eps (inf if never / unverified)."""
    if not verified:
        return float("inf")
    dt = result_record.get("dist_trajectory", [])
    st = result_record.get("shot_trajectory", [])
    for d, s in zip(dt, st):
        if d <= eps:
            return float(s)
    return float("inf")


def queries_to_threshold(result_record: dict, verified: bool, eps: float) -> float:
    if not verified:
        return float("inf")
    dt = result_record.get("dist_trajectory", [])
    qt = result_record.get("query_trajectory", [])
    for d, q in zip(dt, qt):
        if d <= eps:
            return float(q)
    return float("inf")


# --------------------------------------------------------------------------- #
# Aggregation over an attacked image set
# --------------------------------------------------------------------------- #
def summarize_attack(perts, successes, *, cap: Optional[float] = None) -> dict:
    """Median/mean perturbation over verified successes + success rate + dispersion."""
    perts = np.asarray(perts, float)
    successes = np.asarray(successes, bool)
    ok = perts[successes & np.isfinite(perts)]
    out = {
        "success_rate": float(np.mean(successes)) if len(successes) else 0.0,
        "n_images": int(len(successes)),
        "n_success": int(len(ok)),
        "median_perturbation": float(np.median(ok)) if len(ok) else float("nan"),
        "mean_perturbation": float(np.mean(ok)) if len(ok) else float("nan"),
        "std_perturbation": float(np.std(ok)) if len(ok) else float("nan"),
    }
    if len(ok):
        out["dispersion"] = float(np.std(ok) / (abs(np.mean(ok)) + 1e-12))  # sigma/|mean|
    if cap is not None:                       # capped median counts failures at `cap`
        filled = np.where(successes & np.isfinite(perts), perts, cap)
        out["median_perturbation_capped"] = float(np.median(filled))
    return out


def robust_accuracy(perts, successes, eps: float) -> float:
    """Fraction of (clean-correct) images with no verified adversarial within eps."""
    perts = np.asarray(perts, float)
    successes = np.asarray(successes, bool)
    broken = successes & np.isfinite(perts) & (perts <= eps)
    return float(1.0 - np.mean(broken)) if len(successes) else float("nan")


def success_rate_at_eps(perts, successes, eps: float) -> float:
    perts = np.asarray(perts, float)
    successes = np.asarray(successes, bool)
    return float(np.mean(successes & np.isfinite(perts) & (perts <= eps)))


# --------------------------------------------------------------------------- #
# Seed aggregation (results.json schema) and correlation tests
# --------------------------------------------------------------------------- #
def seed_aggregate(values) -> dict:
    """{'mean','std','seeds':[...]} -- the plan's headline results schema."""
    v = np.asarray([x for x in values if x is not None and np.isfinite(x)], float)
    if len(v) == 0:
        return {"mean": float("nan"), "std": float("nan"), "seeds": list(values)}
    return {"mean": float(np.mean(v)), "std": float(np.std(v)),
            "sem": float(np.std(v) / np.sqrt(len(v))), "seeds": [float(x) for x in values]}


def pearson_with_p(x, y) -> dict:
    """Pearson r with p-value (correlation claims must report both, plan Sec. 6)."""
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return {"r": float("nan"), "p": float("nan"), "n": int(m.sum())}
    r, p = stats.pearsonr(x[m], y[m])
    return {"r": float(r), "p": float(p), "n": int(m.sum())}


# --------------------------------------------------------------------------- #
# Significance tests for head-to-head claims (plan Sec. 6: report effect + p-value)
# --------------------------------------------------------------------------- #
def bootstrap_ci(values, stat="median", n_boot=5000, alpha=0.05, seed=0) -> dict:
    """Percentile bootstrap CI for a statistic of finite values (e.g. median pert)."""
    v = np.asarray([x for x in values if np.isfinite(x)], float)
    if len(v) < 2:
        return {"point": float("nan"), "lo": float("nan"), "hi": float("nan"), "n": int(len(v))}
    rng = np.random.default_rng(seed)
    fn = np.median if stat == "median" else np.mean
    boot = np.array([fn(rng.choice(v, size=len(v), replace=True)) for _ in range(n_boot)])
    return {"point": float(fn(v)), "lo": float(np.quantile(boot, alpha / 2)),
            "hi": float(np.quantile(boot, 1 - alpha / 2)), "n": int(len(v)), "stat": stat}


def compare_perturbations(perts_a, perts_b) -> dict:
    """Mann-Whitney U on two attacks' per-image perturbations (successes only).

    Answers 'does attack A reach smaller perturbations than B?' with an effect size
    (rank-biserial) and a p-value, over the images each actually broke.
    """
    a = np.asarray([x for x in perts_a if np.isfinite(x)], float)
    b = np.asarray([x for x in perts_b if np.isfinite(x)], float)
    if len(a) < 3 or len(b) < 3:
        return {"p_value": float("nan"), "n_a": int(len(a)), "n_b": int(len(b))}
    u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    rank_biserial = 1.0 - 2.0 * u / (len(a) * len(b))     # effect size in [-1, 1]
    return {"median_a": float(np.median(a)), "median_b": float(np.median(b)),
            "u_statistic": float(u), "p_value": float(p),
            "rank_biserial_effect": float(rank_biserial),
            "n_a": int(len(a)), "n_b": int(len(b))}


def proportion_test(succ_a, n_a, succ_b, n_b) -> dict:
    """Two-proportion z-test for success-rate differences (e.g. calibrated vs baseline)."""
    if n_a == 0 or n_b == 0:
        return {"p_value": float("nan")}
    pa, pb = succ_a / n_a, succ_b / n_b
    pooled = (succ_a + succ_b) / (n_a + n_b)
    se = np.sqrt(pooled * (1 - pooled) * (1 / n_a + 1 / n_b))
    z = (pa - pb) / se if se > 0 else 0.0
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return {"prop_a": float(pa), "prop_b": float(pb), "diff": float(pa - pb),
            "z": float(z), "p_value": float(p), "n_a": int(n_a), "n_b": int(n_b)}


In [ ]:
%%writefile hlq/concentration.py
"""Concentration diagnostics -- the RQ5 confound guardrail (plan Sec. 3, T6).

Exponential concentration [25,26] shrinks a VQC's output margin as qubit count grows.
A model deep in concentration is nearly *data-independent*: it looks spectacularly
"robust" (an attacker cannot move a decision value that barely varies) while being
useless as a classifier.  Every robustness number in this study is therefore
reported alongside:

* ``var_f``            -- variance of the decision value over the data (the
                          concentration order parameter; decays ~ exp(-b n));
* ``above_chance``     -- clean accuracy minus chance, the denominator that makes
                          a robustness claim meaningful;
* ``trivially_robust`` -- the T6 flag: clean accuracy at chance => report as
                          "trivially robust", never as "defended".
"""
from __future__ import annotations

import numpy as np
from scipy import stats

CHANCE = 0.5


def margin_stats(clf, X) -> dict:
    """Decision-value spread over a dataset: the concentration order parameter."""
    f = clf.decision_function(np.asarray(X))
    return {
        "var_f": float(np.var(f)),
        "std_f": float(np.std(f)),
        "mean_abs_f": float(np.mean(np.abs(f))),
        "median_abs_f": float(np.median(np.abs(f))),
        "max_abs_f": float(np.max(np.abs(f))),
    }


def trivially_robust(clean_acc: float, tol: float = 0.02) -> bool:
    """T6: at-chance clean accuracy means robustness is an artifact, not a defense."""
    return bool(clean_acc <= CHANCE + tol)


def robustness_guardrail(clean_acc: float, median_pert: float) -> dict:
    """Report robustness *relative to clean accuracy above chance* (H5)."""
    above = float(clean_acc - CHANCE)
    return {
        "clean_accuracy": float(clean_acc),
        "above_chance": above,
        "trivially_robust": trivially_robust(clean_acc),
        # perturbation per unit of genuine (above-chance) predictive power
        "robustness_per_above_chance": (float(median_pert) / above
                                        if above > 1e-6 and np.isfinite(median_pert)
                                        else float("nan")),
    }


def fit_exponential_concentration(ns, var_f) -> dict:
    """Fit Var[f] ~ exp(-b n): linear regression of log(Var) on n, with R^2 and p."""
    ns = np.asarray(ns, float)
    v = np.asarray(var_f, float)
    m = np.isfinite(v) & (v > 0)
    if m.sum() < 3:
        return {"decay_rate_b": float("nan"), "r_squared": float("nan"),
                "p_value": float("nan"), "n_points": int(m.sum())}
    res = stats.linregress(ns[m], np.log(v[m]))
    return {
        "decay_rate_b": float(-res.slope),          # Var ~ exp(-b n)
        "intercept": float(res.intercept),
        "r_squared": float(res.rvalue ** 2),
        "p_value": float(res.pvalue),
        "n_points": int(m.sum()),
        "model": "log(Var[f]) = intercept - b*n",
    }


def fit_inverse_n(ns, values) -> dict:
    """Fit y ~ a + c/n (the plan's 1/n scaling; requires R^2 > 0.95 to extrapolate)."""
    ns = np.asarray(ns, float)
    y = np.asarray(values, float)
    m = np.isfinite(y) & (ns > 0)
    if m.sum() < 3:
        return {"r_squared": float("nan"), "n_points": int(m.sum())}
    res = stats.linregress(1.0 / ns[m], y[m])
    return {
        "slope_c": float(res.slope), "intercept_a": float(res.intercept),
        "r_squared": float(res.rvalue ** 2), "p_value": float(res.pvalue),
        "n_points": int(m.sum()),
        "extrapolation_allowed": bool(res.rvalue ** 2 > 0.95),   # plan Sec. 6 gate
        "model": "y = a + c/n",
    }


In [ ]:
%%writefile hlq/analysis.py
"""Derived analyses: budget curves, the RQ2 interior optimum, and model validation.

Complements :mod:`hlq.metrics` (per-run metrics) and :mod:`hlq.concentration`
(RQ5 diagnostics) -- this module turns collections of runs into the paper's claims.

``validate_p_flip_on_model`` is the second half of the double-step validation: T1
checks the closed-form flip model against Monte-Carlo at *synthetic* margins; this
checks it against the *real trained VQC's* margins, and reports Pearson r with a
p-value as the plan requires (|r| > 0.9 demands p < 0.001).
"""
from __future__ import annotations

import json
import os

import numpy as np

from .metrics import pearson_with_p
from .oracle import p_flip, sample_label


# --------------------------------------------------------------------------- #
# JSON I/O
# --------------------------------------------------------------------------- #
def save_json(obj, path):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, "w") as fh:
        json.dump(obj, fh, indent=2, default=_default)
    return path


def _default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.bool_,)):
        return bool(o)
    raise TypeError(f"not JSON serialisable: {type(o)}")


def load_json(path):
    with open(path) as fh:
        return json.load(fh)


# --------------------------------------------------------------------------- #
# RQ2: interior optimum of the query/shot split
# --------------------------------------------------------------------------- #
def find_interior_optimum(xs, ys) -> dict:
    """Locate argmin of a budget-allocation curve and test whether it is *interior*.

    H2 predicts an interior optimum: too few shots -> coin-flip labels swamp the
    normal estimate; too many -> too few probes/queries to converge.  A boundary
    optimum is a legitimate (reportable) negative result for H2.
    """
    xs = np.asarray(xs, float)
    ys = np.asarray(ys, float)
    m = np.isfinite(ys)
    if m.sum() < 3:
        return {"is_interior": False, "reason": "insufficient finite points",
                "n_points": int(m.sum())}
    xs_f, ys_f = xs[m], ys[m]
    j = int(np.argmin(ys_f))
    interior = 0 < j < len(ys_f) - 1
    return {
        "argmin_x": float(xs_f[j]), "min_y": float(ys_f[j]),
        "is_interior": bool(interior),
        "index": j, "n_points": int(len(ys_f)),
        "edge_values": [float(ys_f[0]), float(ys_f[-1])],
        # how much better the optimum is than the worse edge (effect size)
        "gain_vs_worst_edge": float(max(ys_f[0], ys_f[-1]) - ys_f[j]),
    }


def budget_curve(records, x_key="total_budget", y_key="median_perturbation") -> dict:
    """Collect (x, y) from a list of aggregated cells, sorted by x."""
    pts = sorted(((r[x_key], r[y_key]) for r in records if np.isfinite(r.get(y_key, np.nan))),
                 key=lambda p: p[0])
    return {"x": [float(a) for a, _ in pts], "y": [float(b) for _, b in pts],
            "x_key": x_key, "y_key": y_key}


def monotone_non_increasing(xs, ys, tol=1e-3) -> bool:
    """T5: larger budget must never yield a *worse* median perturbation (within noise)."""
    order = np.argsort(np.asarray(xs, float))
    y = np.asarray(ys, float)[order]
    y = y[np.isfinite(y)]
    return bool(np.all(np.diff(y) <= tol))


# --------------------------------------------------------------------------- #
# Born-model validation on the real trained classifier
# --------------------------------------------------------------------------- #
def validate_p_flip_on_model(clf, X, shots=(10, 30, 100, 300, 1000), trials=400,
                             seed=0, max_points=200) -> dict:
    """Predicted vs empirical flip rate at the REAL model's margins (double-step step 2).

    For each test point the exact ``f`` gives a predicted ``p_flip``; the empirical
    rate comes from independently sampling ``trials`` S-shot labels.  Reports Pearson
    r with p-value per shot count and pooled.
    """
    rng = np.random.default_rng(seed)
    X = np.asarray(X)[:max_points]
    f = clf.decision_function(X)
    true_lab = np.where(f >= 0, 1, -1)
    per_shot, pred_all, emp_all = {}, [], []
    for S in shots:
        pred = p_flip(f, int(S))
        emp = np.empty(len(f))
        for i, fi in enumerate(f):
            labs = sample_label(np.full(trials, fi), int(S), rng)
            emp[i] = np.mean(labs != true_lab[i])
        per_shot[str(S)] = {
            "correlation": pearson_with_p(pred, emp),
            "mean_abs_error": float(np.mean(np.abs(pred - emp))),
            "predicted": pred.tolist(), "empirical": emp.tolist(),
        }
        pred_all.append(pred)
        emp_all.append(emp)
    pooled = pearson_with_p(np.concatenate(pred_all), np.concatenate(emp_all))
    return {
        "test": "p_flip_model_vs_empirical_on_trained_VQC",
        "shots": list(shots), "n_points": int(len(f)), "trials_per_point": int(trials),
        "per_shot": per_shot, "pooled_correlation": pooled,
        "margins": np.abs(f).tolist(),
        # plan Sec. 6 gate: |r| > 0.9 requires p < 0.001
        "passed": bool(abs(pooled["r"]) > 0.9 and pooled["p"] < 1e-3),
    }


In [ ]:
%%writefile hlq/attacks/__init__.py
"""Decision-based (hard-label) attacks and the white-box / transfer references.

The HSJA family (calibrated, fixed, PopSkipJump, classical anchor) all inherit the
single skeleton in :mod:`hlq.attacks.base`; each overrides only its shot-budget and
side-of-boundary *policy*.  PGD (white-box) and transfer are separate references.
"""
from .base import AttackResult, DecisionBasedAttack, Domain
from .hsja_fixed import FixedShotHSJA
from .hsja_quantum import CalibratedHSJA
from .popskipjump import PopSkipJump
from .pgd_whitebox import pgd_attack
from .momentum import momentum_attack
from .transfer import transfer_attack

BUILDERS = {
    "calibrated_hsja": CalibratedHSJA,
    "fixed_hsja": FixedShotHSJA,
    "popskipjump": PopSkipJump,
    "classical_hsja": FixedShotHSJA,   # same skeleton, deterministic oracle (S=None)
}

__all__ = [
    "AttackResult", "DecisionBasedAttack", "Domain",
    "FixedShotHSJA", "CalibratedHSJA", "PopSkipJump",
    "pgd_attack", "momentum_attack", "transfer_attack", "BUILDERS",
]


In [ ]:
%%writefile hlq/attacks/base.py
"""Shared decision-based attack skeleton (HopSkipJump geometry, plan Sec. 4.3).

All hard-label variants inherit :class:`DecisionBasedAttack`.  The skeleton -- init
-> boundary binary-search -> Monte-Carlo normal estimate -> geometric step -> repeat
-- lives here exactly once.  Concrete attacks override only *policy* hooks:

* ``_decide_adversarial(x, phase)`` : the side-of-boundary decision and how many
  shots / repeats it costs  (this is where the stochastic oracle bites -- M1);
* ``_grad_budget(t)``               : the (B, S_probe) split of the per-iteration
  gradient budget  (M2).

Every side-of-boundary decision goes through ``_decide_adversarial`` so the
stochastic-oracle handling is defined in one place per attack, never duplicated.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable, Optional

import numpy as np

from ..config import AttackConfig
from ..oracle import BudgetExhausted, StochasticOracle


@dataclass
class Domain:
    """Valid region for perturbed inputs (the encoder's input space)."""

    kind: str = "box"           # "box" (clip to [lo,hi]) | "sphere" (L2-normalise) | "none"
    lo: float = 0.0
    hi: float = float(np.pi)

    def project(self, X: np.ndarray) -> np.ndarray:
        if self.kind == "box":
            return np.clip(X, self.lo, self.hi)
        if self.kind == "sphere":
            n = np.linalg.norm(X, axis=-1, keepdims=True)
            n = np.where(n == 0, 1.0, n)
            return X / n
        return X


@dataclass
class AttackResult:
    success: bool
    perturbation: float                       # final L2 (np.inf if never adversarial)
    x_adv: np.ndarray
    queries: int
    shots: int
    dist_trajectory: list = field(default_factory=list)
    query_trajectory: list = field(default_factory=list)
    shot_trajectory: list = field(default_factory=list)
    meta: dict = field(default_factory=dict)

    def to_record(self) -> dict:
        r = dict(success=bool(self.success), perturbation=float(self.perturbation),
                 queries=int(self.queries), shots=int(self.shots),
                 dist_trajectory=[float(d) for d in self.dist_trajectory],
                 query_trajectory=[int(q) for q in self.query_trajectory],
                 shot_trajectory=[int(s) for s in self.shot_trajectory])
        r.update(self.meta)
        return r


class DecisionBasedAttack:
    """HopSkipJump-style ell_2 untargeted attack; policy hooks left abstract."""

    def __init__(self, cfg: AttackConfig):
        self.cfg = cfg

    # --------------------------------------------------------------------- #
    # Policy hooks (overridden by subclasses)
    # --------------------------------------------------------------------- #
    def _decide_adversarial(self, x: np.ndarray, phase: str) -> bool:
        raise NotImplementedError

    def _grad_budget(self, t: int, x_b: np.ndarray, x0: np.ndarray) -> tuple:
        """Return (B, S_probe) for the normal estimate at iteration t.

        ``x_b`` (current boundary point) and ``x0`` are passed so a calibrated
        policy can estimate the local probe margin; fixed policies ignore them.
        """
        return self.cfg.num_probes, self.cfg.probe_shots

    # convenience shared by fixed/popskipjump policies
    def _labels_batch(self, X: np.ndarray, S: Optional[int]) -> np.ndarray:
        return self._oracle.label_batch(X, S)

    # --------------------------------------------------------------------- #
    # Shared geometry
    # --------------------------------------------------------------------- #
    def _binary_search(self, x_adv: np.ndarray, x0: np.ndarray) -> np.ndarray:
        """Project to the boundary by bisecting the segment [x0 (safe), x_adv (adv)].

        Invariant: ``hi`` is always an accepted-adversarial alpha, ``lo`` is not, so the
        returned point is on the adversarial side.  A policy may set ``_halt_bisect``
        to stop early once the oracle can no longer resolve the side of the boundary
        at the affordable shot budget (the calibrated attack's resolution limit, M1).
        """
        lo, hi = 0.0, 1.0
        tol = self.cfg.bin_search_tol
        self._halt_bisect = False
        while hi - lo > tol:
            mid = 0.5 * (lo + hi)
            xm = self._domain.project(x0 + mid * (x_adv - x0))
            if self._decide_adversarial(xm, "bsearch"):
                hi = mid
            else:
                lo = mid
            if self._halt_bisect:            # resolution limit reached; hi stays valid
                break
        return self._domain.project(x0 + hi * (x_adv - x0))

    def _estimate_gradient(self, x_b, x0, t, rng) -> np.ndarray:
        """HSJA Monte-Carlo boundary-normal estimate with batched probes (M2)."""
        d = x_b.size
        B, S_probe = self._grad_budget(t, x_b, x0)
        dist = float(np.linalg.norm(x_b - x0))
        delta = max(dist / d, 10 * self.cfg.bin_search_tol)      # HSJA ell_2 probe radius
        u = rng.normal(size=(B, d))
        u /= np.linalg.norm(u, axis=1, keepdims=True)
        X = self._domain.project(x_b[None, :] + delta * u)
        labels = self._labels_batch(X, S_probe)
        phi = np.where(labels != self._y0, 1.0, -1.0)            # +1 if adversarial
        phi = phi - phi.mean()                                   # baseline control variate
        g = (phi[:, None] * u).mean(axis=0)
        ng = np.linalg.norm(g)
        if ng < 1e-12:                                           # degenerate: random dir
            g = rng.normal(size=d)
            ng = np.linalg.norm(g)
        return g / ng

    def _geometric_step(self, x_b, x0, grad, t) -> np.ndarray:
        """Move along the estimated normal, halving until still adversarial."""
        dist = float(np.linalg.norm(x_b - x0))
        eps = dist / np.sqrt(t + 1.0)
        for _ in range(25):
            cand = self._domain.project(x_b + eps * grad)
            if self._decide_adversarial(cand, "step"):
                return cand
            eps *= 0.5
            if eps < self.cfg.bin_search_tol:
                break
        return x_b                                               # no improving step found

    def _init_adv(self, x0, y0, init_pool, rng) -> Optional[np.ndarray]:
        """Find a starting adversarial point: nearest opposite-class ref, else random."""
        if init_pool is not None and len(init_pool):
            order = np.argsort(np.linalg.norm(init_pool - x0[None, :], axis=1))
            for idx in order[: min(len(order), 40)]:
                if self._decide_adversarial(init_pool[idx], "init"):
                    return init_pool[idx].copy()
        for _ in range(200):                                     # random fallback
            if self._domain.kind == "sphere":
                c = rng.normal(size=x0.size)
            else:
                c = rng.uniform(self._domain.lo, self._domain.hi, size=x0.size)
            c = self._domain.project(c)
            if self._decide_adversarial(c, "init"):
                return c
        return None

    # --------------------------------------------------------------------- #
    # Main loop
    # --------------------------------------------------------------------- #
    def attack(self, oracle: StochasticOracle, x0, y0, init_pool, rng,
               domain: Domain) -> AttackResult:
        self._oracle = oracle
        self._y0 = int(y0)
        self._domain = domain
        oracle.budget = min(oracle.budget, self.cfg.total_budget)   # exact fixed-T stop
        x0 = np.asarray(x0, dtype=np.float64)

        current = x0.copy()
        cur_dist = np.inf
        best_dist = np.inf                    # recorded for reference only (see note)
        dist_traj, q_traj, s_traj = [], [], []
        interrupted = False
        try:
            # T4 trivial-case guard: if x0 is already adversarial there is nothing to do.
            if self._decide_adversarial(x0, "init"):
                return AttackResult(True, 0.0, x0.copy(), oracle.n_queries,
                                    oracle.n_shots, dist_trajectory=[0.0],
                                    query_trajectory=[oracle.n_queries],
                                    shot_trajectory=[oracle.n_shots],
                                    meta={"already_adversarial": True})
            x_adv = self._init_adv(x0, y0, init_pool, rng)
            if x_adv is None:
                return AttackResult(False, np.inf, x0.copy(), oracle.n_queries,
                                    oracle.n_shots, meta={"init_failed": True})
            x_b = self._binary_search(x_adv, x0)
            current, cur_dist = x_b.copy(), float(np.linalg.norm(x_b - x0))
            best_dist = cur_dist
            dist_traj, q_traj, s_traj = [cur_dist], [oracle.n_queries], [oracle.n_shots]

            for t in range(self.cfg.iterations):
                grad = self._estimate_gradient(x_b, x0, t, rng)
                x_step = self._geometric_step(x_b, x0, grad, t)
                x_b = self._binary_search(x_step, x0)
                current = x_b.copy()
                cur_dist = float(np.linalg.norm(x_b - x0))
                best_dist = min(best_dist, cur_dist)
                dist_traj.append(cur_dist)
                q_traj.append(oracle.n_queries)
                s_traj.append(oracle.n_shots)
        except BudgetExhausted:
            interrupted = True

        # NOTE: we return the FINAL iterate, as standard HopSkipJump does -- never the
        # minimum-*believed*-distance iterate.  Under a stochastic oracle each accepted
        # boundary point carries a ~delta chance of being a false positive, and a false
        # positive always lies much closer to x0; taking the minimum would therefore
        # systematically select the attack's own errors and return a point that is not
        # truly adversarial.
        return AttackResult(
            success=np.isfinite(cur_dist), perturbation=cur_dist, x_adv=current,
            queries=oracle.n_queries, shots=oracle.n_shots,
            dist_trajectory=dist_traj, query_trajectory=q_traj, shot_trajectory=s_traj,
            meta={"budget_interrupted": interrupted, "best_believed_distance": best_dist},
        )


In [ ]:
%%writefile hlq/attacks/hsja_fixed.py
"""Naive HSJA port: fixed shots everywhere, no Born-rule calibration (baseline).

This is the "port an attack mechanically" control (plan Sec. 4.4).  Every decision
uses a constant ``S`` and every gradient estimate a constant ``(B, S_probe)``.
With ``fixed_shots=None`` the oracle is deterministic, giving the *classical
anchor* (HSJA on a classical NN) for free from the same skeleton.
"""
from __future__ import annotations

from .base import DecisionBasedAttack


class FixedShotHSJA(DecisionBasedAttack):
    def _decide_adversarial(self, x, phase) -> bool:
        S = self.cfg.init_eval_shots if phase == "init" else self.cfg.fixed_shots
        return int(self._oracle.label(x, S)) != self._y0

    def _grad_budget(self, t, x_b, x0) -> tuple:
        return self.cfg.num_probes, self.cfg.probe_shots


In [ ]:
%%writefile hlq/attacks/hsja_quantum.py
"""Calibrated hard-label attack -- the paper's core method (plan Sec. 4.3, M1+M2).

Two Born-rule-calibrated modifications over the naive HSJA port.

**M1 -- shot-calibrated boundary search.**  A hard-label binary search drives its
midpoints *onto* the boundary, where the margin ``m -> 0`` and the Born-rule flip
probability ``-> 1/2``: the sign there is unresolvable at *any* finite shot budget,
and a single false-positive "adversarial" reading collapses the search toward ``x0``
and never recovers (the noise-induced inward bias).  We therefore do not attempt the
impossible.  The shot budget defines a *boundary-localization resolution*

    tau(S) = z_delta * sqrt( (1 - f^2) / S )      (z_delta = Phi^{-1}(1 - delta)),

the Born-rule standard error of the S-shot estimator.  A point is declared
adversarial only when its estimate clears that resolution (``|f_hat| > tau``);
inside the band the decision is "too close to call" and is resolved *conservatively*
(treated as safe), which keeps the search outside the true boundary.  Shots are spent
**adaptively**: a cheap pilot settles easy, high-margin points immediately, and the
budget is escalated (doubling) only for points near the boundary, up to a cap.  This
is what a uniform-shot scheme (PopSkipJump) cannot do -- it pays the same price
everywhere.

**M2 -- budget-aware normal estimation.**  Given the per-iteration gradient budget
``T_grad``, the ``(B, S_probe)`` split is chosen by
:func:`hlq.budget.optimal_probe_split` from the locally estimated probe margin,
realising the interior optimum of RQ2.  ``force_probe_shots`` / ``force_num_probes``
let the RQ2 driver sweep the split by hand to trace the empirical curve.
"""
from __future__ import annotations

import numpy as np
from scipy.stats import norm

from ..budget import optimal_probe_split
from .base import DecisionBasedAttack

_Z_CACHE = {}


def _z(delta: float) -> float:
    """Phi^{-1}(1-delta), memoised: scipy's ppf is slow and this runs per decision."""
    z = _Z_CACHE.get(delta)
    if z is None:
        z = float(norm.ppf(1.0 - delta))
        _Z_CACHE[delta] = z
    return z


class CalibratedHSJA(DecisionBasedAttack):
    def __init__(self, cfg, decision_cap: int = None):
        super().__init__(cfg)
        self.force_probe_shots = None      # set by RQ2 sweep to force S_probe
        self.force_num_probes = None
        self._decision_cap_override = decision_cap
        self._m2_log = []                  # per-iteration (B, S_probe, margin) diagnostics
        self._decision_shots = []          # per-decision shot spend (M1 diagnostics)
        self._pilot_rng = np.random.default_rng(cfg.seed + 99)

    # -- M1: Born-calibrated adaptive shot escalation ----------------------- #
    def _pilot_shots(self) -> int:
        return int(max(8, min(32, self.cfg.fixed_shots // 3)))

    def _decision_cap(self) -> int:
        """Finest affordable resolution: tau_min ~ z/sqrt(cap). Bounds one decision's cost."""
        if self._decision_cap_override is not None:
            return int(self._decision_cap_override)
        return int(max(512, self.cfg.total_budget // 50))

    def _decide_adversarial(self, x, phase) -> bool:
        delta = self.cfg.delta_decision
        if phase == "init":
            delta = min(delta, 0.02)                    # very reliable initialisation
        z = _z(delta)
        cap = self._decision_cap()
        S_tot, f_acc, S = 0, 0.0, self._pilot_shots()
        while True:
            f_new = self._oracle.estimate_f(x, S)       # independent Binomial draw at same f
            f_acc = (f_acc * S_tot + f_new * S) / (S_tot + S)   # pooled estimate
            S_tot += S
            tau = z * np.sqrt(max(1.0 - f_acc * f_acc, 1e-6) / S_tot)
            if abs(f_acc) > tau:                        # resolved at confidence 1-delta
                self._decision_shots.append(S_tot)
                return (1 if f_acc >= 0 else -1) != self._y0
            if S_tot >= cap:
                # Unresolvable at the affordable budget: we have localised the boundary
                # to the shot-limited resolution.  Answer conservatively (treat as safe)
                # and stop bisecting -- pushing further only burns shots on coin flips.
                self._decision_shots.append(S_tot)
                self._halt_bisect = True
                return False
            S = min(S_tot, cap - S_tot)                 # escalate: double the budget

    # -- M2: budget-aware (B, S_probe) split -------------------------------- #
    def _estimate_probe_margin(self, x_b, x0) -> float:
        """Cheap pilot estimate of the typical |f| a probe sees at this boundary point."""
        d = x_b.size
        dist = float(np.linalg.norm(x_b - x0))
        delta_r = max(dist / d, 10 * self.cfg.bin_search_tol)
        u = self._pilot_rng.normal(size=(5, d))
        u /= np.linalg.norm(u, axis=1, keepdims=True)
        X = self._domain.project(x_b[None, :] + delta_r * u)
        fhat = self._oracle.estimate_f_batch(X, self._pilot_shots())
        return float(np.clip(np.median(np.abs(fhat)), 1e-3, 0.999))

    def _grad_budget(self, t, x_b, x0) -> tuple:
        T_iter = self.cfg.total_budget / max(1, self.cfg.iterations)
        T_grad = int(max(64, self.cfg.grad_budget_frac * T_iter))
        if self.force_probe_shots is not None:              # RQ2 manual sweep
            S = int(self.force_probe_shots)
            B = int(self.force_num_probes) if self.force_num_probes else max(8, T_grad // S)
            self._m2_log.append({"t": t, "S_probe": S, "B": B, "forced": True})
            return B, S
        margin = self._estimate_probe_margin(x_b, x0)
        B, S, _ = optimal_probe_split(T_grad, margin)
        self._m2_log.append({"t": t, "S_probe": S, "B": B, "margin_est": margin})
        return B, S

    def attack(self, *a, **k):
        res = super().attack(*a, **k)
        res.meta["m2_log"] = self._m2_log
        if self._decision_shots:
            res.meta["mean_decision_shots"] = float(np.mean(self._decision_shots))
            res.meta["n_decisions"] = int(len(self._decision_shots))
        return res


In [ ]:
%%writefile hlq/attacks/popskipjump.py
"""PopSkipJump baseline: decision-based attack for a *constant*-noise oracle [15].

The key comparison for RQ3.  PopSkipJump makes each decision reliable by repeating
the query and majority-voting, with the repeat count fixed from a single *global*
flip-rate estimate ``p0`` -- it is oracle-agnostic and does **not** use the Born-rule
margin structure.  Consequences at equal total budget: it over-spends far from the
boundary (where one shot would suffice) and under-spends near it (where the true
flip rate ~ 1/2 exceeds the global ``p0`` its repeat count was calibrated for).
The calibrated attack should beat it (H3).
"""
from __future__ import annotations

import numpy as np
from scipy.stats import binom

from .base import DecisionBasedAttack


def repeats_for_target(p0: float, delta: float, r_max: int = 51) -> int:
    """Smallest odd R with majority-vote error P(Bin(R,p0) > R/2) <= delta."""
    for R in range(1, r_max + 1, 2):
        if binom.sf(R // 2, R, p0) <= delta:
            return R
    return r_max


class PopSkipJump(DecisionBasedAttack):
    def __init__(self, cfg):
        super().__init__(cfg)
        self._p0 = None
        self._R = None

    def _calibrate_global_noise(self):
        """Fix the repeat count from the ASSUMED constant flip rate p0.

        This is the baseline's defining limitation and must not be estimated at a
        convenient point: measuring the flip rate far from the boundary returns ~0 and
        collapses the method to a single unrepeated query (i.e. the naive fixed-shot
        port).  PopSkipJump's premise is a constant, oracle-agnostic noise level, so we
        give it exactly that -- one repeat count applied uniformly everywhere.
        """
        self._p0 = float(min(max(self.cfg.psj_assumed_p0, 0.02), 0.45))
        self._R = repeats_for_target(self._p0, self.cfg.delta_decision)

    def _reliable_label(self, x) -> int:
        if self._p0 is None:
            self._calibrate_global_noise()
        S = self.cfg.fixed_shots
        labs = np.array([int(self._oracle.label(x, S)) for _ in range(self._R)])
        return 1 if labs.sum() >= 0 else -1

    def _decide_adversarial(self, x, phase) -> bool:
        return self._reliable_label(x) != self._y0

    def _grad_budget(self, t, x_b, x0) -> tuple:
        return self.cfg.num_probes, self.cfg.probe_shots

    def attack(self, *a, **k):
        res = super().attack(*a, **k)
        res.meta["popskipjump_p0"] = self._p0
        res.meta["popskipjump_repeats"] = self._R
        return res


In [ ]:
%%writefile hlq/attacks/pgd_whitebox.py
"""White-box PGD reference (plan Sec. 4.4, upper-bound attacker).

Differentiates the classifier through the circuit (parameter-shift / backprop), so it
is the strongest possible adversary and yields a *lower bound* on the achievable
perturbation against which the gradient-free hard-label attacks are measured.  We
report the minimal-norm adversarial: descend ``y0 * f`` until the label flips, then
bisect back toward ``x0`` using the exact (infinite-shot) sign to minimise ‖x'-x0‖.
"""
from __future__ import annotations

import numpy as np


def pgd_attack(vqc, x0, y0, domain, cfg) -> dict:
    import torch

    x0 = np.asarray(x0, dtype=np.float64)
    x0t = torch.tensor(x0, dtype=torch.float64)
    y0 = int(y0)
    x = x0t.clone()
    step = cfg.pgd_step_size
    grad_steps = 0
    x_adv = None

    # Phase 1: gradient descent on y0 * f until the label flips (find any adversarial).
    for _ in range(cfg.pgd_steps):
        x = x.detach().requires_grad_(True)
        f = vqc.decision_function_torch(x.unsqueeze(0))[0]
        (y0 * f).backward()
        grad_steps += 1
        g = x.grad.detach()
        g = g / (g.norm() + 1e-12)
        x = x.detach() - step * g
        x = torch.tensor(domain.project(x.numpy()), dtype=torch.float64)
        f_now = float(vqc.decision_function(x.numpy().reshape(1, -1))[0])
        if (1 if f_now >= 0 else -1) != y0:
            x_adv = x.numpy().copy()
            break

    if x_adv is None:
        return {"success": False, "perturbation": float("inf"), "x_adv": x0.copy(),
                "queries": 0, "shots": 0, "grad_steps": grad_steps}

    # Phase 2: exact-sign bisection toward x0 for the minimal-norm crossing.
    lo, hi = 0.0, 1.0
    for _ in range(40):
        mid = 0.5 * (lo + hi)
        xm = domain.project(x0 + mid * (x_adv - x0))
        f = float(vqc.decision_function(xm.reshape(1, -1))[0])
        if (1 if f >= 0 else -1) != y0:
            hi = mid
        else:
            lo = mid
    x_b = domain.project(x0 + hi * (x_adv - x0))
    dist = float(np.linalg.norm(x_b - x0))
    return {"success": True, "perturbation": dist, "x_adv": x_b,
            "queries": 0, "shots": 0, "grad_steps": grad_steps}


In [ ]:
%%writefile hlq/attacks/momentum.py
"""Momentum-based quantum adversarial attack -- the QMI precedent [6].

The direct comparison the plan requires: a white-box, momentum-boosted iterative attack
(MI-FGSM, Dong et al. 2018) differentiated *through the circuit* (backprop /
parameter-shift). Like the PGD reference it is a white-box upper bound -- it computes a
gradient of the classifier, which the realistic hard-label threat model forbids -- but it
is the state-of-the-art gradient attacker for VQCs and so anchors where the calibrated
hard-label attack sits relative to the strongest published quantum adversary.

Momentum accumulates an L1-normalised gradient velocity
``g <- mu * g + grad / ||grad||_1``, which stabilises the direction and escapes poor
local optima -- the paper's advantage over plain (P)GD. We report the *minimal-norm*
adversarial (descend ``y0 * f`` under momentum until the label flips, then bisect back
toward ``x0`` with the exact sign), identically to :func:`hlq.attacks.pgd_whitebox.pgd_attack`
so the two white-box references are measured on the same footing.
"""
from __future__ import annotations

import numpy as np


def momentum_attack(vqc, x0, y0, domain, cfg) -> dict:
    import torch

    x0 = np.asarray(x0, dtype=np.float64)
    x0t = torch.tensor(x0, dtype=torch.float64)
    y0 = int(y0)
    x = x0t.clone()
    g = torch.zeros_like(x0t)                       # momentum (velocity) accumulator
    mu = float(getattr(cfg, "momentum_mu", 1.0))
    step = cfg.pgd_step_size
    grad_steps = 0
    x_adv = None

    # Phase 1: momentum gradient descent on y0 * f until the label flips.
    for _ in range(cfg.pgd_steps):
        x = x.detach().requires_grad_(True)
        f = vqc.decision_function_torch(x.unsqueeze(0))[0]
        (y0 * f).backward()
        grad_steps += 1
        gr = x.grad.detach()
        g = mu * g + gr / (gr.abs().sum() + 1e-12)   # MI-FGSM: L1-normalised accumulation
        x = x.detach() - step * g / (g.norm() + 1e-12)
        x = torch.tensor(domain.project(x.numpy()), dtype=torch.float64)
        f_now = float(vqc.decision_function(x.numpy().reshape(1, -1))[0])
        if (1 if f_now >= 0 else -1) != y0:
            x_adv = x.numpy().copy()
            break

    if x_adv is None:
        return {"success": False, "perturbation": float("inf"), "x_adv": x0.copy(),
                "queries": 0, "shots": 0, "grad_steps": grad_steps}

    # Phase 2: exact-sign bisection toward x0 for the minimal-norm crossing.
    lo, hi = 0.0, 1.0
    for _ in range(40):
        mid = 0.5 * (lo + hi)
        xm = domain.project(x0 + mid * (x_adv - x0))
        f = float(vqc.decision_function(xm.reshape(1, -1))[0])
        if (1 if f >= 0 else -1) != y0:
            hi = mid
        else:
            lo = mid
    x_b = domain.project(x0 + hi * (x_adv - x0))
    return {"success": True, "perturbation": float(np.linalg.norm(x_b - x0)),
            "x_adv": x_b, "queries": 0, "shots": 0, "grad_steps": grad_steps}


In [ ]:
%%writefile hlq/attacks/transfer.py
"""Classical-surrogate transfer attack (plan Sec. 4.4, alternative black-box route).

Query the VQC to label a pool, fit a small classical MLP surrogate, run white-box
PGD on the surrogate, and transfer the perturbation to the VQC.  Contextualises the
query cost of the decision-based attacks: transfer is cheap in queries but its
perturbations are typically larger / less reliable because the surrogate only
approximates the quantum boundary.
"""
from __future__ import annotations

import numpy as np


def _train_surrogate(X, y01, seed=0, epochs=150):
    import torch

    torch.manual_seed(seed)
    d = X.shape[1]
    net = torch.nn.Sequential(
        torch.nn.Linear(d, 32), torch.nn.Tanh(),
        torch.nn.Linear(32, 32), torch.nn.Tanh(),
        torch.nn.Linear(32, 1),
    ).double()
    Xt = torch.tensor(X, dtype=torch.float64)
    yt = torch.tensor(y01, dtype=torch.float64).reshape(-1, 1)
    opt = torch.optim.Adam(net.parameters(), lr=0.01)
    for _ in range(epochs):
        opt.zero_grad()
        logit = net(Xt)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logit, yt)
        loss.backward()
        opt.step()
    return net


def transfer_attack(vqc, oracle, x0, y0, X_pool, domain, cfg, rng) -> dict:
    import torch

    x0 = np.asarray(x0, dtype=np.float64)
    y0 = int(y0)

    # 1. label the pool with hard-label VQC queries (charged)
    y_pool = np.array([oracle.label(xp, cfg.fixed_shots) for xp in X_pool])
    pool_queries = oracle.n_queries
    net = _train_surrogate(X_pool, (y_pool + 1) // 2, seed=cfg.seed)

    # 2. white-box PGD on the surrogate to flip x0's surrogate label
    x = torch.tensor(x0, dtype=torch.float64)
    target_orig = torch.tensor([float((y0 + 1) // 2)], dtype=torch.float64)
    for _ in range(cfg.pgd_steps):
        x = x.detach().requires_grad_(True)
        logit = net(x.unsqueeze(0))[0, 0]
        # ascend the loss wrt the ORIGINAL label -> push x away from y0 (flip it)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logit.unsqueeze(0),
                                                                     target_orig)
        loss.backward()
        g = x.grad.detach()
        x = x.detach() + cfg.pgd_step_size * g / (g.norm() + 1e-12)
        x = torch.tensor(domain.project(x.numpy()), dtype=torch.float64)

    x_pert = x.numpy()
    # 3. transfer + exact-sign bisection toward x0 for the minimal transferable crossing
    f = float(vqc.decision_function(x_pert.reshape(1, -1))[0])
    oracle.n_queries += 1
    if (1 if f >= 0 else -1) == y0:                           # transfer failed to flip
        return {"success": False, "perturbation": float("inf"), "x_adv": x0.copy(),
                "queries": oracle.n_queries, "shots": oracle.n_shots,
                "pool_queries": int(pool_queries)}
    lo, hi = 0.0, 1.0
    for _ in range(30):
        mid = 0.5 * (lo + hi)
        xm = domain.project(x0 + mid * (x_pert - x0))
        fm = float(vqc.decision_function(xm.reshape(1, -1))[0])
        oracle.n_queries += 1
        if (1 if fm >= 0 else -1) != y0:
            hi = mid
        else:
            lo = mid
    x_b = domain.project(x0 + hi * (x_pert - x0))
    return {"success": True, "perturbation": float(np.linalg.norm(x_b - x0)),
            "x_adv": x_b, "queries": oracle.n_queries, "shots": oracle.n_shots,
            "pool_queries": int(pool_queries)}


In [ ]:
%%writefile hlq/defenses/__init__.py
"""Inference-time defenses, re-evaluated for the FIRST time against a gradient-free
hard-label attacker (plan Sec. 4.5, RQ4).

Both defenses wrap a trained :class:`hlq.classifier.VQC` and expose the same
``decision_function`` interface, so the oracle and every attack run on them unchanged.
"""
from ..config import DefenseConfig
from .noise import NoisyVQC
from .randomized_encoding import RandomizedEncodingVQC


def wrap_defense(vqc, defense: DefenseConfig):
    """Return (defended_classifier, is_stochastic) for the requested defense."""
    if defense.name == "none":
        return vqc, False
    if defense.name == "depolarizing":
        return NoisyVQC(vqc, defense), False          # deterministic f given x
    if defense.name == "randomized_encoding":
        return RandomizedEncodingVQC(vqc, defense), bool(defense.randomized_per_query)
    raise ValueError(defense.name)


__all__ = ["NoisyVQC", "RandomizedEncodingVQC", "wrap_defense"]


In [ ]:
%%writefile hlq/defenses/noise.py
"""Quantum-noise defense: depolarizing channels injected at inference [16].

Faithful density-matrix model (``default.mixed``): a depolarizing channel of
strength ``p`` follows every ansatz layer (and the encoding).  Noise acts *during*
computation, so ``f_noisy(x)`` is not a global rescaling of ``f_ideal`` -- it
reshapes the boundary and shrinks margins non-uniformly, which is exactly the effect
whose gradient-free robustness we test (RQ4) and whose margin-collapse must be told
apart from genuine defense (RQ5).

Cost note: density-matrix simulation is ``O(4**n)``; the defense experiments cap ``n``.
"""
from __future__ import annotations

import numpy as np
import pennylane as qml

from ..classifier import _observable, apply_encoding
from ..config import DefenseConfig


class NoisyVQC:
    def __init__(self, base_vqc, defense: DefenseConfig):
        self.cfg = base_vqc.cfg
        self.weights = base_vqc.weights
        self.p = float(defense.depolarizing_p)
        self._qnode = None
        self._batched = None            # broadcasting support auto-detected on first call

    def _build(self):
        if self._qnode is not None:
            return self._qnode
        cfg, p = self.cfg, self.p
        dev = qml.device("default.mixed", wires=cfg.n_qubits)

        def circ(x, w):
            wires = range(cfg.n_qubits)
            if cfg.encoding == "reuploading":
                for r in range(cfg.reupload_layers):
                    qml.AngleEmbedding(x, wires=wires, rotation="Y")
                    qml.StronglyEntanglingLayers(w[r : r + 1], wires=wires)
                    for i in wires:
                        qml.DepolarizingChannel(p, wires=i)
            else:
                apply_encoding(x, cfg)
                for l in range(w.shape[0]):                 # per-layer noise injection
                    qml.StronglyEntanglingLayers(w[l : l + 1], wires=wires)
                    for i in wires:
                        qml.DepolarizingChannel(p, wires=i)
            return qml.expval(_observable(cfg))

        self._qnode = qml.QNode(circ, dev)
        return self._qnode

    def decision_function(self, X: np.ndarray) -> np.ndarray:
        X = np.atleast_2d(np.asarray(X, dtype=np.float64))
        qn = self._build()
        if self._batched is None:                           # detect broadcasting once
            try:
                out = np.asarray(qn(X, self.weights), dtype=np.float64).reshape(-1)
                self._batched = len(out) == len(X)
                if self._batched:
                    return np.clip(out, -1.0, 1.0)
            except Exception:
                self._batched = False
        if self._batched:
            out = np.asarray(qn(X, self.weights), dtype=np.float64).reshape(-1)
        else:
            out = np.array([float(qn(x, self.weights)) for x in X])
        return np.clip(out, -1.0, 1.0)

    def predict(self, X):
        return np.where(self.decision_function(X) >= 0, 1, -1).astype(np.int64)

    def clean_accuracy(self, X, y):
        return float(np.mean(self.predict(X) == np.asarray(y)))

    def margins(self, X):
        return np.abs(self.decision_function(X))


In [ ]:
%%writefile hlq/defenses/randomized_encoding.py
"""Randomized-encoding defense: fresh random rotations per query [17].

A data-independent random rotation layer is inserted after the encoding and
resampled on every query.  For the legitimate user the effect averages out; for the
attacker each query's decision value ``f`` is perturbed by an *independent* random
amount, so the boundary the attacker probes jitters from call to call -- an
attacker-side barren plateau on top of the shot noise.  ``randomized_strength``
tunes the plateau intensity (and trades against clean accuracy).

This is pure-state (fast, lightning), so it scales to all ``n``.  The per-query
randomness makes the oracle mark the classifier ``stochastic`` (no caching of ``f``).
"""
from __future__ import annotations

import numpy as np
import pennylane as qml

from ..classifier import _observable, apply_ansatz, apply_encoding
from ..config import DefenseConfig


class RandomizedEncodingVQC:
    def __init__(self, base_vqc, defense: DefenseConfig):
        self.cfg = base_vqc.cfg
        self.weights = base_vqc.weights
        self.strength = float(defense.randomized_strength)
        self.per_query = bool(defense.randomized_per_query)
        self._rng = np.random.default_rng(self.cfg.seed + 4242)
        self._fixed = None                    # used when per_query is False
        self._qnode = None

    def _build(self):
        if self._qnode is not None:
            return self._qnode
        cfg = self.cfg
        try:
            dev = qml.device("lightning.qubit", wires=cfg.n_qubits)
        except Exception:
            dev = qml.device("default.qubit", wires=cfg.n_qubits)

        def circ(x, w, rand):
            wires = range(cfg.n_qubits)
            if cfg.encoding == "reuploading":
                for r in range(cfg.reupload_layers):
                    qml.AngleEmbedding(x, wires=wires, rotation="Y")
                    qml.StronglyEntanglingLayers(w[r : r + 1], wires=wires)
            else:
                apply_encoding(x, cfg)
            for i in wires:                                  # random per-query layer
                qml.RY(rand[..., i], wires=i)
            if cfg.encoding != "reuploading":
                apply_ansatz(w, cfg)
            return qml.expval(_observable(cfg))

        self._qnode = qml.QNode(circ, dev)
        return self._qnode

    def _sample_rand(self, batch):
        if self.per_query:
            return self._rng.normal(0.0, self.strength, size=(batch, self.cfg.n_qubits))
        if self._fixed is None:
            self._fixed = self._rng.normal(0.0, self.strength, size=(self.cfg.n_qubits,))
        return np.broadcast_to(self._fixed, (batch, self.cfg.n_qubits))

    def decision_function(self, X: np.ndarray) -> np.ndarray:
        X = np.atleast_2d(np.asarray(X, dtype=np.float64))
        qn = self._build()
        rand = self._sample_rand(len(X))
        out = np.asarray(qn(X, self.weights, rand), dtype=np.float64).reshape(-1)
        return np.clip(out, -1.0, 1.0)

    def predict(self, X):
        return np.where(self.decision_function(X) >= 0, 1, -1).astype(np.int64)

    def clean_accuracy(self, X, y):
        return float(np.mean(self.predict(X) == np.asarray(y)))

    def margins(self, X):
        return np.abs(self.decision_function(X))


In [ ]:
%%writefile hlq/runner.py
"""The experiment cell: one (model x defense x attack x seed) -> verified metrics.

Every research question in the plan is a *composition* of cells, so this is the only
place that knows how to run an attack end-to-end.  Cells are the unit of parallelism
(the driver maps them across processes); they take plain configs, so workers rebuild
from cache rather than pickling live QNodes.
"""
from __future__ import annotations

from dataclasses import replace

import numpy as np

from .attacks import BUILDERS, Domain, momentum_attack, pgd_attack, transfer_attack
from .classical import MatchedClassicalNN
from .classifier import train_or_load
from .concentration import margin_stats, robustness_guardrail
from .config import AttackConfig, ClassifierConfig, DefenseConfig
from .data import load_dataset
from .defenses import wrap_defense
from .metrics import (queries_to_threshold, robust_accuracy, shots_to_threshold,
                      summarize_attack, verify)
from .oracle import StochasticOracle
from .seeds import set_seed


def build_domain(cfg: ClassifierConfig) -> Domain:
    """Valid input region for the encoder: angles live in [0, pi]; amplitudes on S^{d-1}."""
    if cfg.encoding == "amplitude":
        return Domain("sphere")
    return Domain("box", 0.0, float(np.pi))


def _verify(clf, x0, x_adv, y0, stochastic: bool, n_samples: int = 25) -> dict:
    """Ground-truth check. For a per-query-stochastic defense the deployed decision is
    the expectation over the randomisation, so average f over ``n_samples`` draws."""
    if not stochastic:
        return verify(clf, x0, x_adv, y0)
    f = float(np.mean([clf.decision_function(np.asarray(x_adv).reshape(1, -1))[0]
                       for _ in range(n_samples)]))
    truly_adv = (1 if f >= 0 else -1) != int(y0)
    return {"verified_success": bool(truly_adv),
            "verified_perturbation": (float(np.linalg.norm(np.asarray(x_adv) - np.asarray(x0)))
                                      if truly_adv else float("inf")),
            "exact_margin_at_adv": abs(f)}


def select_attack_images(clf, bundle, n_images: int, stochastic=False) -> np.ndarray:
    """Indices of the first ``n_images`` test points the DEPLOYED model gets right.

    Attacking already-misclassified points is meaningless, so the evaluation set is
    always clean-correct under the model actually being attacked (defense included).
    """
    pred = clf.predict(bundle.X_test)
    idx = np.where(pred == bundle.y_test)[0]
    return idx[:n_images]


def run_cell(clf_cfg: ClassifierConfig, def_cfg: DefenseConfig, atk_cfg: AttackConfig,
             n_images: int = 250, eps: float = 0.5, force_probe_shots=None,
             verbose: bool = False) -> dict:
    """Run one full experiment cell and return a JSON-ready record."""
    bundle = load_dataset(clf_cfg)
    domain = build_domain(clf_cfg)

    # -- model under attack (classical anchor swaps in a matched NN) ---------- #
    if atk_cfg.name == "classical_hsja":
        model = MatchedClassicalNN(clf_cfg)
        info = model.fit(bundle.X_train, bundle.y_train)
        info["test_acc"] = model.clean_accuracy(bundle.X_test, bundle.y_test)
        deployed, stochastic = model, False
        # A classical NN returns a DETERMINISTIC label: there is no Born rule and no
        # shot noise to sample. Sampling labels from its output would invent a
        # stochastic oracle that does not exist and destroy the anchor's meaning --
        # the anchor exists precisely to show what the walk costs WITHOUT shot noise.
        atk_cfg = replace(atk_cfg, fixed_shots=None, probe_shots=None,
                          init_eval_shots=None, total_budget=10 ** 15)
    else:
        model, info = train_or_load(clf_cfg, bundle)
        deployed, stochastic = wrap_defense(model, def_cfg)

    clean_acc = deployed.clean_accuracy(bundle.X_test, bundle.y_test)
    idxs = select_attack_images(deployed, bundle, n_images, stochastic)

    records, perts, succs = [], [], []
    for i, idx in enumerate(idxs):
        x0, y0 = bundle.X_test[idx], int(bundle.y_test[idx])
        pool = bundle.X_train[bundle.y_train != y0][:40]
        rng = set_seed(atk_cfg.seed * 100003 + int(idx))       # per-image reproducibility

        if atk_cfg.name in ("pgd_whitebox", "momentum"):
            if not hasattr(deployed, "decision_function_torch"):
                continue                                        # white-box needs gradients
            wb = pgd_attack if atk_cfg.name == "pgd_whitebox" else momentum_attack
            res = wb(deployed, x0, y0, domain, atk_cfg)
            v = _verify(deployed, x0, res["x_adv"], y0, stochastic)
            rec = {"queries": res["queries"], "shots": res["shots"],
                   "grad_steps": res.get("grad_steps", 0)}
        elif atk_cfg.name == "transfer":
            orc = StochasticOracle(deployed, rng, stochastic=stochastic)
            res = transfer_attack(deployed, orc, x0, y0, bundle.X_train[:300],
                                  domain, atk_cfg, rng)
            v = _verify(deployed, x0, res["x_adv"], y0, stochastic)
            rec = {"queries": res["queries"], "shots": res["shots"]}
        else:
            orc = StochasticOracle(deployed, rng, stochastic=stochastic,
                                   budget=atk_cfg.total_budget)
            atk = BUILDERS[atk_cfg.name](atk_cfg)
            if force_probe_shots is not None and hasattr(atk, "force_probe_shots"):
                atk.force_probe_shots = force_probe_shots       # RQ2 allocation sweep
            out = atk.attack(orc, x0, y0, pool, rng, domain)
            v = _verify(deployed, x0, out.x_adv, y0, stochastic)
            rec = out.to_record()
            rec["circuit_evals"] = int(orc.n_circuit_evals)
            # cost-to-reach-eps, computed from the trajectory before it is discarded
            rec["queries_to_eps"] = queries_to_threshold(rec, v["verified_success"], eps)
            rec["shots_to_eps"] = shots_to_threshold(rec, v["verified_success"], eps)
            if not verbose:      # trajectories are large; keep them only when asked
                for k in ("dist_trajectory", "query_trajectory", "shot_trajectory",
                          "m2_log"):
                    rec.pop(k, None)

        rec.update(image_index=int(idx), **v)
        records.append(rec)
        perts.append(v["verified_perturbation"])
        succs.append(v["verified_success"])

    def _median_finite(key):
        v = np.array([r.get(key, np.inf) for r in records], float)
        v = v[np.isfinite(v)]
        return float(np.median(v)) if len(v) else float("nan")

    summary = summarize_attack(perts, succs, cap=None)
    summary.update(
        robust_accuracy_at_eps=robust_accuracy(perts, succs, eps),
        eps=eps,
        # plan Sec. 6: cost to REACH the eps threshold (inf/excluded if never reached)
        median_queries_to_eps=_median_finite("queries_to_eps"),
        median_shots_to_eps=_median_finite("shots_to_eps"),
        frac_reaching_eps=float(np.mean([np.isfinite(r.get("queries_to_eps", np.inf))
                                         for r in records])) if records else float("nan"),
        clean_accuracy=clean_acc,
        train_test_acc=info.get("test_acc", float("nan")),
        median_queries=float(np.median([r["queries"] for r in records])) if records else float("nan"),
        median_shots=float(np.median([r["shots"] for r in records])) if records else float("nan"),
    )
    summary.update(robustness_guardrail(clean_acc, summary["median_perturbation"]))
    summary.update(margin_stats(deployed if not stochastic else model, bundle.X_test[:400]))
    return {
        "classifier": clf_cfg.to_dict(), "defense": def_cfg.to_dict(),
        "attack": atk_cfg.to_dict(), "n_images": int(len(idxs)),
        "force_probe_shots": force_probe_shots,
        "summary": summary, "per_image": records,
    }


In [ ]:
%%writefile experiments/run_sanity.py
"""Sanity checks T1-T6, defined before the code (plan Sec. 7).

The double-step validation gate: T1-T4 must pass before any attack result is trusted.

  T1  Born-rule flip model vs Monte-Carlo, and vs PennyLane's own shot sampling
  T2  infinite-shot limit of the calibrated attack -> deterministic HSJA
  T3  attack on a synthetic dataset with an analytically known boundary
  T4  already-adversarial input -> attack returns it unchanged
  T5  budget monotonicity: larger T never yields a worse perturbation
  T6  concentration guard: at-chance clean accuracy is flagged "trivially robust"

Run:  python experiments/run_sanity.py [--quick]
"""
from __future__ import annotations

import argparse
import json
import os
import sys

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from hlq.analysis import monotone_non_increasing, save_json, validate_p_flip_on_model
from hlq.attacks import CalibratedHSJA, Domain, FixedShotHSJA
from hlq.classifier import VQC, train_or_load
from hlq.classifier import weight_shape as vqc_weight_shape
from hlq.concentration import margin_stats, trivially_robust
from hlq.config import AttackConfig, ClassifierConfig
from hlq.data import load_dataset, make_linear_boundary
from hlq.metrics import verify
from hlq.oracle import StochasticOracle, sample_label, validate_p_flip
from hlq.seeds import set_seed

RESULTS = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "results")


# --------------------------------------------------------------------------- #
class LinearOracleModel:
    """Analytic classifier with an exactly known boundary: f(x) = tanh(w.x + b).

    Used by T3: the true point-to-boundary distance of the *sign* boundary is
    |w.x + b| / ||w||, so the attack's recovered perturbation has a ground truth.
    """

    def __init__(self, w, b, scale=1.0):
        self.w = np.asarray(w, float)
        self.b = float(b)
        self.scale = float(scale)

    def decision_function(self, X):
        X = np.atleast_2d(np.asarray(X, float))
        return np.tanh(self.scale * (X @ self.w + self.b))

    def predict(self, X):
        return np.where(self.decision_function(X) >= 0, 1, -1).astype(np.int64)

    def true_distance(self, x):
        return abs(float(x @ self.w + self.b)) / float(np.linalg.norm(self.w))


# --------------------------------------------------------------------------- #
def t1_flip_model(quick=False) -> dict:
    """T1: closed form vs Monte-Carlo, plus a cross-check against PennyLane shots."""
    out = validate_p_flip(trials=4000 if quick else 20000, seed=0)

    # Independent cross-check: our exact-Binomial oracle must reproduce PennyLane's
    # own shot-based sampling for the same state (they must agree in distribution).
    import pennylane as qml

    n, S, reps = 3, 200, 400
    rng = np.random.default_rng(0)
    dev_a = qml.device("default.qubit", wires=n, shots=S)
    dev_e = qml.device("default.qubit", wires=n)
    angles = rng.uniform(0, np.pi, size=n)

    def _c():
        qml.AngleEmbedding(angles, wires=range(n), rotation="Y")
        return qml.expval(qml.PauliZ(0))

    f_exact = float(qml.QNode(_c, dev_e)())
    pl_labels = np.array([1 if float(qml.QNode(_c, dev_a)()) >= 0 else -1 for _ in range(reps)])
    our_labels = sample_label(np.full(reps, f_exact), S, rng)
    p_pl = float(np.mean(pl_labels == 1))
    p_our = float(np.mean(our_labels == 1))
    se = float(np.sqrt(max(p_pl * (1 - p_pl), 1e-9) / reps) + np.sqrt(max(p_our * (1 - p_our), 1e-9) / reps))
    out["pennylane_crosscheck"] = {
        "f_exact": f_exact, "p_plus_pennylane": p_pl, "p_plus_ours": p_our,
        "abs_diff": abs(p_pl - p_our), "tolerance_3se": 3 * se + 0.02,
        "agrees": bool(abs(p_pl - p_our) <= 3 * se + 0.02),
    }
    out["passed"] = bool(out["passed"] and out["pennylane_crosscheck"]["agrees"])
    return out


def t2_infinite_shot_limit(quick=False) -> dict:
    """T2: as S -> infinity the calibrated attack reproduces deterministic HSJA.

    A single global shot-scale sigma raises BOTH shot budgets that make the oracle
    stochastic -- the boundary-decision cap (which sets the resolution tau ~ z/sqrt(cap))
    and the probe shots -- while the probe COUNT and the total budget are held fixed, so
    sigma is the only thing varying.  The budget is deliberately unbounded here: this
    test isolates the S->inf limit, not budget economics (that is RQ2), and a finite cap
    would truncate the walk and confound the two.
    """
    cfg = ClassifierConfig(n_qubits=4, n_layers=2, dataset="synthetic", epochs=12, seed=0)
    b = load_dataset(cfg)
    vqc, _ = train_or_load(cfg, b)
    dom = Domain("box", 0.0, float(np.pi))
    idxs = np.where(vqc.predict(b.X_test) == b.y_test)[0][: (4 if quick else 8)]
    UNBOUNDED = 10 ** 15
    B_PROBES = 100

    def _run(mk, acfg):
        """Per-image perturbations (paired across conditions on the same x0)."""
        ps = []
        for idx in idxs:
            x0, y0 = b.X_test[idx], int(b.y_test[idx])
            pool = b.X_train[b.y_train != y0][:30]
            rng = set_seed(0)
            orc = StochasticOracle(vqc, rng, budget=UNBOUNDED)
            r = mk(acfg).attack(orc, x0, y0, pool, rng, dom)
            v = verify(vqc, x0, r.x_adv, y0)
            ps.append(v["verified_perturbation"])
        return np.array(ps, float)

    def _paired_gap(a, ref):
        """Median |a - ref| / ref over images where BOTH succeeded.

        Paired per-image: the perturbation varies by ~2x across x0, so comparing
        medians of different small samples measures image variance, not convergence.
        """
        m = np.isfinite(a) & np.isfinite(ref)
        if not m.any():
            return float("nan")
        return float(np.median(np.abs(a[m] - ref[m]) / np.maximum(ref[m], 1e-9)))

    # Reference: the deterministic (infinite-shot) oracle -- standard HSJA.
    det_cfg = AttackConfig(iterations=12, total_budget=UNBOUNDED, fixed_shots=None,
                           init_eval_shots=None, num_probes=B_PROBES, probe_shots=None,
                           seed=0)
    det_per_image = _run(FixedShotHSJA, det_cfg)
    det_ok = det_per_image[np.isfinite(det_per_image)]
    det_pert = float(np.median(det_ok)) if len(det_ok) else float("nan")
    det_succ = float(np.mean(np.isfinite(det_per_image)))

    # The scales must be large: a probe sits ~delta from the boundary where |f| ~ 0.03,
    # and p_flip(0.03, S) only approaches 0 for S >> 1/|f|^2 ~ 1e3.  Shots are free in the
    # analytic Born oracle (one Binomial draw per query), so sigma can sweep decades.
    scales = [1, 100, 10_000] if quick else [1, 10, 100, 1_000, 10_000]
    caps, probe_shots, perts, succs, gaps = [], [], [], [], []
    for s in scales:
        cap = 500 * s
        sp = 100 * s
        acfg = AttackConfig(iterations=12, total_budget=UNBOUNDED, delta_decision=0.05,
                            fixed_shots=100, num_probes=B_PROBES, probe_shots=sp, seed=0)

        def _mk(c, cap=cap, sp=sp):
            a = CalibratedHSJA(c, decision_cap=cap)
            a.force_probe_shots = sp              # scale probe shots with sigma too
            a.force_num_probes = B_PROBES         # hold the probe COUNT fixed
            return a

        per_image = _run(_mk, acfg)
        ok = per_image[np.isfinite(per_image)]
        caps.append(cap)
        probe_shots.append(sp)
        perts.append(float(np.median(ok)) if len(ok) else float("nan"))
        succs.append(float(np.mean(np.isfinite(per_image))))
        gaps.append(_paired_gap(per_image, det_per_image))

    return {
        "test": "T2_infinite_shot_limit",
        "deterministic_hsja": {"median_perturbation": det_pert, "success_rate": det_succ,
                               "per_image": det_per_image.tolist()},
        "shot_scales": scales, "decision_caps": caps, "probe_shots": probe_shots,
        "calibrated_median_perturbation": perts, "calibrated_success_rate": succs,
        "paired_relative_gap": gaps,
        "gap_shrinks_with_shots": bool(gaps[-1] <= gaps[0] + 1e-6),
        # The test is CONVERGENCE: the paired per-image gap to deterministic HSJA must
        # shrink into tolerance as shots grow. The absolute success rate over a handful
        # of images is sample-noisy (2/4 = 0.5 is not a failure signal), so it is only a
        # weak floor here, not the criterion.
        "passed": bool(np.isfinite(gaps[-1]) and gaps[-1] < 0.15
                       and gaps[-1] <= gaps[0] + 1e-6 and succs[-1] >= 0.5),
    }


def t3_known_boundary(quick=False) -> dict:
    """T3: recovered perturbation ~ true point-to-boundary distance (analytic model)."""
    n = 4
    w, b0 = make_linear_boundary(n, seed=0)
    model = LinearOracleModel(w, b0, scale=3.0)
    dom = Domain("box", 0.0, float(np.pi))
    rng0 = np.random.default_rng(1)
    X = rng0.uniform(0, np.pi, size=(40, n))
    X = X[np.abs(X @ w + b0) > 0.2][: (4 if quick else 10)]

    acfg = AttackConfig(iterations=25, total_budget=10 ** 12, fixed_shots=None,
                        init_eval_shots=None, num_probes=150, probe_shots=None, seed=0)
    rec, rel = [], []
    for x0 in X:
        y0 = int(model.predict(x0.reshape(1, -1))[0])
        pool = rng0.uniform(0, np.pi, size=(400, n))
        pool = pool[model.predict(pool) != y0][:30]
        rng = set_seed(0)
        orc = StochasticOracle(model, rng, budget=10 ** 12)
        r = FixedShotHSJA(acfg).attack(orc, x0, y0, pool, rng, dom)
        truth = model.true_distance(x0)
        rec.append({"recovered": float(r.perturbation), "true_distance": float(truth),
                    "rel_error": float(abs(r.perturbation - truth) / max(truth, 1e-9))})
        rel.append(rec[-1]["rel_error"])
    med_rel = float(np.median(rel))
    return {"test": "T3_known_boundary", "per_point": rec,
            "median_relative_error": med_rel, "tolerance": 0.15,
            "passed": bool(med_rel < 0.15)}


def t4_already_adversarial(quick=False) -> dict:
    """T4: an input that is already adversarial must be returned unchanged."""
    cfg = ClassifierConfig(n_qubits=4, n_layers=2, dataset="synthetic", epochs=12, seed=0)
    b = load_dataset(cfg)
    vqc, _ = train_or_load(cfg, b)
    dom = Domain("box", 0.0, float(np.pi))
    # points the model gets WRONG: relative to the true label they are already adversarial
    wrong = np.where(vqc.predict(b.X_test) != b.y_test)[0][: (3 if quick else 8)]
    acfg = AttackConfig(iterations=10, total_budget=200000, delta_decision=0.05,
                        fixed_shots=100, num_probes=60, probe_shots=60, seed=0)
    perts, flags = [], []
    for idx in wrong:
        x0, y0 = b.X_test[idx], int(b.y_test[idx])      # y0 = TRUE label; model disagrees
        pool = b.X_train[b.y_train != y0][:20]
        rng = set_seed(0)
        orc = StochasticOracle(vqc, rng, budget=200000)
        r = CalibratedHSJA(acfg).attack(orc, x0, y0, pool, rng, dom)
        perts.append(float(r.perturbation))
        flags.append(bool(r.meta.get("already_adversarial", False)))
    return {"test": "T4_already_adversarial", "n_points": int(len(wrong)),
            "perturbations": perts, "flagged_trivial": flags,
            "all_zero_perturbation": bool(all(p == 0.0 for p in perts)),
            "passed": bool(len(wrong) > 0 and all(flags) and all(p == 0.0 for p in perts))}


def t5_budget_monotonicity(quick=False) -> dict:
    """T5: a larger measurement budget must never give a worse median perturbation."""
    cfg = ClassifierConfig(n_qubits=4, n_layers=2, dataset="synthetic", epochs=12, seed=0)
    b = load_dataset(cfg)
    vqc, _ = train_or_load(cfg, b)
    dom = Domain("box", 0.0, float(np.pi))
    idxs = np.where(vqc.predict(b.X_test) == b.y_test)[0][: (4 if quick else 10)]
    budgets = [20000, 60000] if quick else [10000, 30000, 100000, 300000]
    per_budget = []
    for T in budgets:
        acfg = AttackConfig(iterations=15, total_budget=T, delta_decision=0.05,
                            fixed_shots=100, num_probes=80, probe_shots=60, seed=0)
        ps = []
        for idx in idxs:
            x0, y0 = b.X_test[idx], int(b.y_test[idx])
            pool = b.X_train[b.y_train != y0][:30]
            rng = set_seed(0)
            orc = StochasticOracle(vqc, rng, budget=T)
            r = CalibratedHSJA(acfg).attack(orc, x0, y0, pool, rng, dom)
            v = verify(vqc, x0, r.x_adv, y0)
            ps.append(v["verified_perturbation"])
        per_budget.append(np.array(ps, float))

    # PAIRED over a COMMON success set: perturbation varies ~10x across images, and the
    # set of images that succeed changes with T, so medians of different subsets measure
    # image variance and selection, not the budget trend.
    M = np.vstack(per_budget)
    common = np.all(np.isfinite(M), axis=0)
    meds_all = [float(np.median(p[np.isfinite(p)])) if np.isfinite(p).any() else float("nan")
                for p in per_budget]
    meds = ([float(np.median(p[common])) for p in per_budget] if common.any()
            else meds_all)
    tol = 0.10 * float(np.nanmax(meds)) if np.isfinite(meds).any() else 0.05
    return {"test": "T5_budget_monotonicity", "budgets": budgets,
            "median_perturbation_common_set": meds,
            "median_perturbation_all_successes": meds_all,
            "n_common_images": int(common.sum()), "n_images": int(len(idxs)),
            "tolerance": tol,
            "monotone_non_increasing": monotone_non_increasing(budgets, meds, tol=tol),
            "passed": bool(common.sum() >= 2
                           and monotone_non_increasing(budgets, meds, tol=tol))}


def t6_concentration_guard(quick=False) -> dict:
    """T6: the RQ5 guardrail must fire on a concentrated, at-chance model -- and only then.

    Exponential concentration is a property of RANDOM/deep circuits, not of a trained
    model on a learnable task (training actively fights it), so the guard is exercised
    against randomly-initialised deep circuits with the global observable Z^{otimes n} --
    the named concentration source.  Their decision value must collapse with n and their
    accuracy fall to chance, at which point the model must be reported "trivially robust"
    rather than "defended".  A trained, above-chance model must NOT be flagged.
    Whether *trained* models concentrate is an empirical question -- that is RQ5, not a
    sanity check.
    """
    ns = [4, 6, 8] if quick else [4, 6, 8, 10, 12]
    rows = []
    for n in ns:
        cfg = ClassifierConfig(n_qubits=n, n_layers=10, dataset="synthetic",
                               observable="global_z", epochs=1, seed=0)
        b = load_dataset(cfg)
        vqc = VQC(cfg)                                     # RANDOM (untrained) weights
        rng = np.random.default_rng(0)
        vqc.weights = rng.uniform(0, 2 * np.pi, size=vqc_weight_shape(cfg))
        acc = vqc.clean_accuracy(b.X_test[:400], b.y_test[:400])
        ms = margin_stats(vqc, b.X_test[:400])
        rows.append({"n_qubits": n, "observable": "global_z", "regime": "random_deep",
                     "clean_accuracy": float(acc),
                     "trivially_robust": trivially_robust(acc), **ms})

    # control: a trained, genuinely above-chance model must NOT be flagged
    cfg_t = ClassifierConfig(n_qubits=4, n_layers=2, dataset="synthetic", epochs=12, seed=0)
    bt = load_dataset(cfg_t)
    vqc_t, _ = train_or_load(cfg_t, bt)
    acc_t = vqc_t.clean_accuracy(bt.X_test, bt.y_test)
    trained = {"n_qubits": 4, "observable": "local_z", "regime": "trained",
               "clean_accuracy": float(acc_t), "trivially_robust": trivially_robust(acc_t),
               **margin_stats(vqc_t, bt.X_test[:400])}

    var = [r["var_f"] for r in rows]
    fired = bool(any(r["trivially_robust"] for r in rows))
    decays = bool(var[-1] < var[0])
    return {"test": "T6_concentration_guard", "random_deep_global_z": rows,
            "trained_control": trained,
            "var_f_collapses_with_n": decays, "var_f": var,
            "guard_fires_on_concentrated_model": fired,
            "guard_silent_on_trained_model": bool(not trained["trivially_robust"]),
            "passed": bool(decays and fired and not trained["trivially_robust"])}


# --------------------------------------------------------------------------- #
def main():
    ap = argparse.ArgumentParser(description="Sanity tests T1-T6")
    ap.add_argument("--quick", action="store_true", help="smaller sweeps for a fast gate")
    ap.add_argument("--only", default=None, help="comma-separated subset, e.g. T1,T3")
    ap.add_argument("--out", default=os.path.join(RESULTS, "sanity.json"))
    args = ap.parse_args()

    tests = {"T1": t1_flip_model, "T2": t2_infinite_shot_limit, "T3": t3_known_boundary,
             "T4": t4_already_adversarial, "T5": t5_budget_monotonicity,
             "T6": t6_concentration_guard}
    if args.only:
        tests = {k: v for k, v in tests.items() if k in args.only.split(",")}

    results = {}
    if args.only and os.path.exists(args.out):     # merge, don't clobber other tests
        with open(args.out) as fh:
            results = {k: v for k, v in json.load(fh).items() if not k.startswith("_")}
    for name, fn in tests.items():
        print(f"[sanity] running {name} ...", flush=True)
        try:
            results[name] = fn(args.quick)
            print(f"[sanity] {name}: passed={results[name].get('passed')}", flush=True)
        except Exception as exc:                     # keep going; record the failure
            results[name] = {"test": name, "passed": False, "error": repr(exc)}
            print(f"[sanity] {name}: ERROR {exc!r}", flush=True)

    gate = all(results[t].get("passed") for t in ("T1", "T2", "T3", "T4") if t in results)
    results["_summary"] = {
        "passed": {k: bool(v.get("passed")) for k, v in results.items()},
        "trust_gate_T1_T4": bool(gate),
        "note": "Plan Sec. 7: code is not trusted until T1-T4 pass.",
    }
    save_json(results, args.out)
    print(f"\n[sanity] wrote {args.out}")
    print(f"[sanity] T1-T4 trust gate: {'PASS' if gate else 'FAIL'}")
    for k, v in results["_summary"]["passed"].items():
        print(f"   {k}: {'PASS' if v else 'FAIL'}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile experiments/driver.py
"""Experiment driver: composes cells into the plan's research questions.

Every RQ is a set of :func:`hlq.runner.run_cell` calls that vary exactly one axis with
principled defaults for the rest (the plan's ablation design, Sec. 4.4).  Cells are the
unit of parallelism.  ``config.json`` is dumped at the start of every run and results
land in ``results/<rq>.json`` with the plan's schema
``{"metric": {"mean":..., "std":..., "seeds":[...]}}``.

Run:  python experiments/driver.py --rq rq1 --preset smoke
      python experiments/driver.py --rq all --preset full --jobs 4
"""
from __future__ import annotations

import argparse
import os
import sys
import time

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import hashlib
import json

from hlq.analysis import _default, budget_curve, find_interior_optimum, save_json
from hlq.classifier import train_or_load
from hlq.concentration import fit_exponential_concentration
from hlq.config import AttackConfig, ClassifierConfig, DefenseConfig
from hlq.data import load_dataset
from hlq.metrics import seed_aggregate
from hlq.runner import run_cell

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RESULTS = os.path.join(ROOT, "results")

# --------------------------------------------------------------------------- #
# Presets: scale knobs only -- the experimental design is identical across them.
# --------------------------------------------------------------------------- #
PRESETS = {
    "smoke":  dict(n_images=4,   seeds=(0, 1),                   iterations=8,  budget=20_000),
    "medium": dict(n_images=40,  seeds=(0, 1, 2),                iterations=15, budget=60_000),
    # "kaggle": tuned to finish RQ4+RQ5 inside a single ~12h CPU session, with real
    # statistics (3 seeds).  The heavy blocks additionally cap their most expensive axis
    # internally (RQ4 -> n=4 density-matrix; RQ5 -> n<=10 by default).
    "kaggle": dict(n_images=60,  seeds=(0, 1, 2),                iterations=20, budget=80_000),
    # "kaggle8": publication-grade statistics (8 seeds) for the head-to-head blocks
    # (RQ1/RQ2/RQ3/RQ4). RQ5 self-caps its seed count (see _RQ5_MAX_SEEDS) because its
    # per-cell cost is model TRAINING, which does not shrink with fewer attacked images.
    "kaggle8": dict(n_images=80,  seeds=(0, 1, 2, 3, 4, 5, 6, 7), iterations=20, budget=80_000),
    "full":   dict(n_images=250, seeds=(0, 1, 2, 3, 4, 5, 6, 7), iterations=30, budget=200_000),
}

# RQ5 qubit ladder. n=12 statevector TRAINING dominates the whole run (tens of minutes
# per model), so it is opt-in via HLQ_RQ5_MAX_N; the default stops at 10.
_RQ5_MAX_N = int(os.environ.get("HLQ_RQ5_MAX_N", "10"))
# RQ5 trains n x observable x seed models serially; at 8 seeds that is 64 models. Cap the
# seed count so RQ5 stays inside a session even under the 8-seed presets (override with
# HLQ_RQ5_MAX_SEEDS). The concentration TREND is a diagnostic, not a head-to-head test,
# so 5 seeds is ample.
_RQ5_MAX_SEEDS = int(os.environ.get("HLQ_RQ5_MAX_SEEDS", "5"))
# RQ4 qubit sweep for the defense re-evaluation. Default n=4 (density-matrix depolarizing
# is O(4^n)); set e.g. HLQ_RQ4_NS="4,6" to test scaling across qubit count.
_RQ4_NS = [int(x) for x in os.environ.get("HLQ_RQ4_NS", "4").split(",")]

# Principled defaults held fixed while one axis varies (plan Sec. 4.1/5).
DEFAULT_N, DEFAULT_L, DEFAULT_ENC, DEFAULT_OBS = 8, 5, "angle", "local_z"
DEFAULT_DATASET = "mnist_3v5"


def _clf(seed, **kw) -> ClassifierConfig:
    base = dict(n_qubits=DEFAULT_N, n_layers=DEFAULT_L, encoding=DEFAULT_ENC,
                observable=DEFAULT_OBS, dataset=DEFAULT_DATASET, seed=seed)
    base.update(kw)
    return ClassifierConfig(**base)


def _atk(name, seed, P, **kw) -> AttackConfig:
    base = dict(name=name, seed=seed, iterations=P["iterations"],
                total_budget=P["budget"], delta_decision=0.05, fixed_shots=100,
                num_probes=80, probe_shots=60)
    base.update(kw)
    return AttackConfig(**base)


# --------------------------------------------------------------------------- #
def warmup_models(clf_cfgs, verbose=True):
    """Train/cache every unique classifier serially BEFORE parallel cells run.

    Parallel workers would otherwise race to write the same model-cache file.
    """
    seen = {}
    for c in clf_cfgs:
        if c.key() in seen or c.dataset == "synthetic" and False:
            continue
        seen[c.key()] = c
    for i, (k, c) in enumerate(seen.items(), 1):
        t0 = time.time()
        b = load_dataset(c)
        _, info = train_or_load(c, b)
        if verbose:
            print(f"  [warmup {i}/{len(seen)}] {k}  test_acc={info.get('test_acc', float('nan')):.3f}"
                  f"  ({time.time()-t0:.1f}s)", flush=True)
    return list(seen.values())


# --------------------------------------------------------------------------- #
# Per-cell checkpointing + resume.  A cell is the unit of work AND of durability:
# each completed cell is written to results/checkpoints/<rq>/<id>.json the instant it
# finishes, so a session that hits Kaggle's 12h wall keeps every cell it computed, and
# re-running skips them.  Workers write distinct files, so there is no contention.
# --------------------------------------------------------------------------- #
_CKPT_DIR = None


def _cell_id(task) -> str:
    payload = {"clf": task["clf_cfg"].to_dict(), "def": task["def_cfg"].to_dict(),
               "atk": task["atk_cfg"].to_dict(), "n_images": task.get("n_images"),
               "force_probe_shots": task.get("force_probe_shots")}
    return hashlib.md5(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]


def _run_cell_ckpt(task, ckpt_dir):
    """Run one cell, or load it from a checkpoint if it was already computed."""
    path = os.path.join(ckpt_dir, _cell_id(task) + ".json")
    if os.path.exists(path):
        try:
            with open(path) as fh:
                return json.load(fh)
        except Exception:
            pass                                   # corrupt/partial -> recompute
    res = run_cell(**task)
    tmp = path + ".tmp"
    with open(tmp, "w") as fh:
        json.dump(res, fh, default=_default)
    os.replace(tmp, path)                          # atomic publish
    return res


def _run_all(jobs, tasks):
    """Execute cells (parallel when jobs>1), checkpointing each when _CKPT_DIR is set."""
    ckpt = _CKPT_DIR
    if ckpt:
        os.makedirs(ckpt, exist_ok=True)
        fn = lambda t: _run_cell_ckpt(t, ckpt)
        done = sum(os.path.exists(os.path.join(ckpt, _cell_id(t) + ".json")) for t in tasks)
        if done:
            print(f"  [resume] {done}/{len(tasks)} cells already checkpointed", flush=True)
    else:
        fn = lambda t: run_cell(**t)
    if jobs <= 1:
        return [fn(t) for t in tasks]
    from joblib import Parallel, delayed
    if ckpt:
        return Parallel(n_jobs=jobs, backend="loky", verbose=5)(
            delayed(_run_cell_ckpt)(t, ckpt) for t in tasks)
    return Parallel(n_jobs=jobs, backend="loky", verbose=5)(
        delayed(run_cell)(**t) for t in tasks)


def _agg(cells, metric):
    return seed_aggregate([c["summary"].get(metric, float("nan")) for c in cells])


def _group(cells, keyfn):
    out = {}
    for c in cells:
        out.setdefault(keyfn(c), []).append(c)
    return out


METRICS = ("success_rate", "median_perturbation", "median_queries", "median_shots",
           "robust_accuracy_at_eps", "clean_accuracy")


def _summarize(groups) -> dict:
    """Aggregate each condition's cells over seeds into the plan's results schema."""
    return {k: {m: _agg(v, m) for m in METRICS} for k, v in groups.items()}


# --------------------------------------------------------------------------- #
# RQ1: feasibility -- can a hard-label attack fool a VQC, and at what query cost?
# --------------------------------------------------------------------------- #
def rq1(P, jobs):
    attacks = ["calibrated_hsja", "fixed_hsja", "popskipjump", "pgd_whitebox",
               "momentum", "transfer", "classical_hsja"]
    tasks, clfs = [], []
    for s in P["seeds"]:
        c = _clf(s)
        clfs.append(c)
        for a in attacks:
            tasks.append(dict(clf_cfg=c, def_cfg=DefenseConfig("none"),
                              atk_cfg=_atk(a, s, P), n_images=P["n_images"]))
    warmup_models(clfs)
    cells = _run_all(jobs, tasks)
    return {"cells": cells,
            "aggregated": _summarize(_group(cells, lambda c: c["attack"]["name"]))}


# --------------------------------------------------------------------------- #
# RQ2: query/shot economics -- budget curve, probe-allocation sweep, M1/M2 split
# --------------------------------------------------------------------------- #
def rq2(P, jobs):
    """RQ2 has three independent sweeps; HLQ_RQ2_PARTS selects which to run.

    They are separable, so an interrupted run can be finished by re-running only the
    missing part (e.g. HLQ_RQ2_PARTS="m1m2") instead of redoing hours of completed work.
    """
    parts = set(os.environ.get("HLQ_RQ2_PARTS", "budget,alloc,m1m2").split(","))
    out = {"parts_run": sorted(parts)}
    clfs = [_clf(s) for s in P["seeds"]]
    warmup_models(clfs)

    # (a) budget curve: perturbation vs total measurement budget T
    budgets = [P["budget"] // 8, P["budget"] // 4, P["budget"] // 2, P["budget"],
               P["budget"] * 2]
    if "budget" not in parts:
        budgets = []
    tasks = [dict(clf_cfg=_clf(s), def_cfg=DefenseConfig("none"),
                  atk_cfg=_atk("calibrated_hsja", s, P, total_budget=T),
                  n_images=P["n_images"])
             for s in P["seeds"] for T in budgets]
    if budgets:
        cells = _run_all(jobs, tasks)
        g = _group(cells, lambda c: c["attack"]["total_budget"])
        agg = _summarize(g)
        out["budget_curve"] = {
            "budgets": budgets, "aggregated": agg,
            "curve": budget_curve([{"total_budget": int(k),
                                    "median_perturbation": v["median_perturbation"]["mean"]}
                                   for k, v in agg.items()]),
            "cells": cells,
        }

    # (b) allocation sweep: perturbation vs FORCED probe shots S_probe at fixed T.
    #     Prop 3 proves the SNR objective is monotone in S, so the optimum should sit at
    #     the smallest shots (a boundary optimum / plateau), not an interior one.
    if "alloc" in parts:
        s_grid = [1, 2, 4, 8, 16, 32, 64, 128, 256]
        tasks = [dict(clf_cfg=_clf(s), def_cfg=DefenseConfig("none"),
                      atk_cfg=_atk("calibrated_hsja", s, P), n_images=P["n_images"],
                      force_probe_shots=sp)
                 for s in P["seeds"] for sp in s_grid]
        cells = _run_all(jobs, tasks)
        g = _group(cells, lambda c: c["force_probe_shots"])
        agg = _summarize(g)
        xs = sorted(g.keys())
        ys = [agg[x]["median_perturbation"]["mean"] for x in xs]
        out["allocation_sweep"] = {
            "probe_shots": xs, "median_perturbation": ys, "aggregated": agg,
            "interior_optimum": find_interior_optimum(xs, ys), "cells": cells,
        }

    # (c) M1/M2 split: fraction of the per-iteration budget given to normal estimation.
    #     Prop 3(b) predicts the genuine interior optimum lives HERE.
    if "m1m2" in parts:
        fracs = [0.1, 0.25, 0.5, 0.75, 0.9]
        tasks = [dict(clf_cfg=_clf(s), def_cfg=DefenseConfig("none"),
                      atk_cfg=_atk("calibrated_hsja", s, P, grad_budget_frac=fr),
                      n_images=P["n_images"])
                 for s in P["seeds"] for fr in fracs]
        cells = _run_all(jobs, tasks)
        g = _group(cells, lambda c: c["attack"]["grad_budget_frac"])
        agg = _summarize(g)
        xs = sorted(g.keys())
        ys = [agg[x]["median_perturbation"]["mean"] for x in xs]
        out["m1_m2_split"] = {
            "grad_budget_frac": xs, "median_perturbation": ys, "aggregated": agg,
            "interior_optimum": find_interior_optimum(xs, ys), "cells": cells,
        }
    return out


# --------------------------------------------------------------------------- #
# RQ3: does Born-rule calibration beat a constant-noise treatment at equal budget?
# --------------------------------------------------------------------------- #
def rq3(P, jobs):
    methods = ["calibrated_hsja", "popskipjump", "fixed_hsja"]
    budgets = [P["budget"] // 4, P["budget"], P["budget"] * 2]
    clfs = [_clf(s) for s in P["seeds"]]
    warmup_models(clfs)
    tasks = [dict(clf_cfg=_clf(s), def_cfg=DefenseConfig("none"),
                  atk_cfg=_atk(m, s, P, total_budget=T), n_images=P["n_images"])
             for s in P["seeds"] for m in methods for T in budgets]
    cells = _run_all(jobs, tasks)
    g = _group(cells, lambda c: (c["attack"]["name"], c["attack"]["total_budget"]))
    return {"cells": cells,
            "aggregated": {f"{k[0]}@T={k[1]}": {m: _agg(v, m) for m in METRICS}
                           for k, v in g.items()},
            "methods": methods, "budgets": budgets}


# --------------------------------------------------------------------------- #
# RQ4: first gradient-free evaluation of the quantum-noise / randomized defenses
# --------------------------------------------------------------------------- #
def rq4(P, jobs):
    defenses = [DefenseConfig("none"),
                DefenseConfig("depolarizing", depolarizing_p=0.05),
                DefenseConfig("randomized_encoding", randomized_strength=0.30)]
    attacks = ["calibrated_hsja", "pgd_whitebox"]
    # The depolarizing defense uses density-matrix simulation (O(4^n), no broadcasting),
    # which dominates cost, so RQ4 runs at n=4 (256-dim) -- ~16x faster than n=6 -- with a
    # reduced image count and a lighter attack budget. The mechanism (margin collapse ->
    # gradient-free robustness) is qubit-count-independent; RQ5 handles the n sweep.
    ns = [min(DEFAULT_N, n) for n in _RQ4_NS]
    n_img = max(8, P["n_images"] // 3)
    def4 = dict(iterations=min(P["iterations"], 15),
                total_budget=min(P["budget"], 40_000))
    clfs = [_clf(s, n_qubits=n) for s in P["seeds"] for n in ns]
    warmup_models(clfs)
    tasks = [dict(clf_cfg=_clf(s, n_qubits=n), def_cfg=d,
                  atk_cfg=_atk(a, s, P, **def4), n_images=n_img)
             for s in P["seeds"] for n in ns for d in defenses for a in attacks]
    cells = _run_all(jobs, tasks)
    # key includes n only when more than one is swept, so single-n stays back-compatible
    def _k(c):
        base = f"{c['defense']['name']}|{c['attack']['name']}"
        return base + (f"|n={c['classifier']['n_qubits']}" if len(ns) > 1 else "")
    g = _group(cells, _k)
    return {"cells": cells, "n_qubits": ns,
            "aggregated": {k: {m: _agg(v, m) for m in METRICS} for k, v in g.items()},
            "note": "depolarizing uses density-matrix (default.mixed, O(4^n)); n and "
                    "image count kept small. Set HLQ_RQ4_NS to sweep qubit count."}


# --------------------------------------------------------------------------- #
# RQ5: is apparent robustness really exponential concentration?
# --------------------------------------------------------------------------- #
def rq5(P, jobs):
    ns = [n for n in (4, 6, 8, 10, 12) if n <= _RQ5_MAX_N]     # n=12 opt-in (see top)
    obs = ["local_z", "global_z"]
    seeds = tuple(P["seeds"])[:_RQ5_MAX_SEEDS]                 # cap: training-bound (see top)
    P = {**P, "seeds": seeds}
    n_img = max(8, P["n_images"] // 4)

    # Training the large-n models is the cost; fewer epochs still exhibit the
    # concentration effect this RQ measures (var[f] collapse for the global observable).
    def _c5(s, n, o):
        return _clf(s, n_qubits=n, observable=o, epochs=min(40, 25 if n >= 8 else 40))

    clfs = [_c5(s, n, o) for s in P["seeds"] for n in ns for o in obs]
    warmup_models(clfs)
    tasks = [dict(clf_cfg=_c5(s, n, o), def_cfg=DefenseConfig("none"),
                  atk_cfg=_atk("calibrated_hsja", s, P), n_images=n_img)
             for s in P["seeds"] for n in ns for o in obs]
    cells = _run_all(jobs, tasks)
    g = _group(cells, lambda c: (c["classifier"]["observable"], c["classifier"]["n_qubits"]))
    agg = {f"{k[0]}|n={k[1]}": {m: _agg(v, m) for m in METRICS} for k, v in g.items()}
    fits = {}
    for o in obs:
        xs = [n for n in ns]
        var = [float(np.mean([c["summary"]["var_f"] for c in g[(o, n)]])) for n in ns]
        fits[o] = {"n_qubits": xs, "var_f": var,
                   "exponential_fit": fit_exponential_concentration(xs, var)}
    return {"cells": cells, "aggregated": agg, "concentration_fits": fits,
            "n_qubits": ns, "observables": obs,
            # report the seeds ACTUALLY used (RQ5 caps them); main() records this in
            # _meta so the stored metadata never overstates the statistics.
            "seeds_used": list(seeds)}


# --------------------------------------------------------------------------- #
# Ablations: encoding (C) and ansatz depth
# --------------------------------------------------------------------------- #
def ablation_encoding(P, jobs):
    encs = ["angle", "amplitude", "reuploading"]
    clfs = [_clf(s, encoding=e) for s in P["seeds"] for e in encs]
    warmup_models(clfs)
    tasks = [dict(clf_cfg=_clf(s, encoding=e), def_cfg=DefenseConfig("none"),
                  atk_cfg=_atk("calibrated_hsja", s, P), n_images=P["n_images"])
             for s in P["seeds"] for e in encs]
    cells = _run_all(jobs, tasks)
    return {"cells": cells,
            "aggregated": _summarize(_group(cells, lambda c: c["classifier"]["encoding"]))}


def ablation_depth(P, jobs):
    Ls = [2, 5, 10]
    clfs = [_clf(s, n_layers=L) for s in P["seeds"] for L in Ls]
    warmup_models(clfs)
    tasks = [dict(clf_cfg=_clf(s, n_layers=L), def_cfg=DefenseConfig("none"),
                  atk_cfg=_atk("calibrated_hsja", s, P), n_images=P["n_images"])
             for s in P["seeds"] for L in Ls]
    cells = _run_all(jobs, tasks)
    return {"cells": cells,
            "aggregated": _summarize(_group(cells, lambda c: c["classifier"]["n_layers"]))}


def ablation_dataset(P, jobs):
    ds = ["mnist_3v5", "mnist_0v1", "fashion_mnist"]
    clfs = [_clf(s, dataset=d) for s in P["seeds"] for d in ds]
    warmup_models(clfs)
    tasks = [dict(clf_cfg=_clf(s, dataset=d), def_cfg=DefenseConfig("none"),
                  atk_cfg=_atk("calibrated_hsja", s, P), n_images=P["n_images"])
             for s in P["seeds"] for d in ds]
    cells = _run_all(jobs, tasks)
    return {"cells": cells,
            "aggregated": _summarize(_group(cells, lambda c: c["classifier"]["dataset"]))}


RQS = {"rq1": rq1, "rq2": rq2, "rq3": rq3, "rq4": rq4, "rq5": rq5,
       "ablation_encoding": ablation_encoding, "ablation_depth": ablation_depth,
       "ablation_dataset": ablation_dataset}


# --------------------------------------------------------------------------- #
def main():
    ap = argparse.ArgumentParser(description="Hard-label quantum attack experiments")
    ap.add_argument("--rq", required=True, choices=list(RQS) + ["all"])
    ap.add_argument("--preset", default="smoke", choices=list(PRESETS))
    ap.add_argument("--jobs", type=int, default=1)
    ap.add_argument("--images", type=int, default=None, help="override images per cell")
    ap.add_argument("--seeds", type=int, default=None, help="override number of seeds")
    ap.add_argument("--out", default=RESULTS)
    ap.add_argument("--force", action="store_true",
                    help="recompute even if results/<rq>.json already exists")
    ap.add_argument("--no-checkpoint", action="store_true",
                    help="disable per-cell checkpointing/resume")
    args = ap.parse_args()

    P = dict(PRESETS[args.preset])
    if args.images:
        P["n_images"] = args.images
    if args.seeds:
        P["seeds"] = tuple(range(args.seeds))

    targets = list(RQS) if args.rq == "all" else [args.rq]
    os.makedirs(args.out, exist_ok=True)
    save_json({"preset": args.preset, "params": {**P, "seeds": list(P["seeds"])},
               "defaults": {"n_qubits": DEFAULT_N, "n_layers": DEFAULT_L,
                            "encoding": DEFAULT_ENC, "observable": DEFAULT_OBS,
                            "dataset": DEFAULT_DATASET},
               "jobs": args.jobs},
              os.path.join(args.out, "config.json"))

    global _CKPT_DIR
    for name in targets:
        path = os.path.join(args.out, f"{name}.json")
        if os.path.exists(path) and not args.force:
            print(f"\n=== {name}: results/{name}.json exists -> skip (use --force to redo) ===",
                  flush=True)
            continue
        _CKPT_DIR = None if args.no_checkpoint else os.path.join(args.out, "checkpoints", name)
        t0 = time.time()
        print(f"\n=== {name} (preset={args.preset}, images={P['n_images']}, "
              f"seeds={len(P['seeds'])}) ===", flush=True)
        res = RQS[name](P, args.jobs)
        # an RQ may use fewer seeds than the preset offers (RQ5 caps them); record what
        # was actually used so the metadata never overstates the statistics
        seeds_used = res.get("seeds_used", list(P["seeds"]))
        res["_meta"] = {"rq": name, "preset": args.preset,
                        "params": {**P, "seeds": list(seeds_used)},
                        "preset_seeds": list(P["seeds"]),
                        "runtime_s": round(time.time() - t0, 1)}
        save_json(res, path)
        print(f"=== {name} done in {res['_meta']['runtime_s']}s -> {path}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile experiments/make_figures.py
"""Research-grade figures from the results JSON (plan Sec. 8).

Design rules applied throughout:
* categorical hues assigned in FIXED order from a validated colorblind-safe palette
  (never cycled), always paired with a second encoding (marker + linestyle) because
  the palette's worst adjacent tritan separation sits in the 6-8 dE band;
* magnitude uses a single-hue sequential ramp; never a rainbow;
* NEVER a dual y-axis -- two measures become two stacked panels sharing an x-axis;
* legend for >=2 series, direct labels where <=4, recessive grid/axes, thin marks;
* the figures carry the result geometrically -- no prose annotations asserting a
  conclusion; only data, error bars (std over seeds), fitted curves and references.

Run:  python experiments/make_figures.py [--results results] [--figures figures]
"""
from __future__ import annotations

import argparse
import json
import os
import sys

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator

from hlq.budget import grad_snr_per_budget
from hlq.oracle import p_flip

# --- validated categorical palette (fixed order) + secondary encodings ------- #
PALETTE = ["#2a78d6", "#008300", "#e87ba4", "#eda100", "#1baf7a", "#eb6834",
           "#4a3aa7", "#e34948"]
MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]
LINESTYLES = ["-", "--", "-.", ":", (0, (3, 1, 1, 1)), (0, (5, 1)), (0, (1, 1)),
              (0, (4, 2, 1, 2))]
SEQ_CMAP = "Blues"                      # single hue, light -> dark
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#8a8985", "#e3e3e0"

# stable slot per method so colour follows the entity, never its rank
METHOD_SLOT = {"calibrated_hsja": 0, "fixed_hsja": 5, "popskipjump": 3,
               "pgd_whitebox": 1, "momentum": 7, "transfer": 6, "classical_hsja": 4}
METHOD_LABEL = {"calibrated_hsja": "Calibrated HSJA (ours, M1+M2)",
                "fixed_hsja": "Fixed-shot HSJA (naive port)",
                "popskipjump": "PopSkipJump (constant noise)",
                "pgd_whitebox": "White-box PGD (reference)",
                "momentum": "Momentum attack [6] (white-box)",
                "transfer": "Classical-surrogate transfer",
                "classical_hsja": "HSJA on matched classical NN"}
# short two-line forms for categorical axes (the long ones collide)
METHOD_SHORT = {"calibrated_hsja": "Calibrated\n(ours)",
                "fixed_hsja": "Fixed-shot\n(naive)",
                "popskipjump": "PopSkipJump\n(const. noise)",
                "pgd_whitebox": "White-box PGD\n(reference)",
                "momentum": "Momentum [6]\n(white-box)",
                "transfer": "Transfer\n(surrogate)",
                "classical_hsja": "Classical NN\n(anchor)"}


def bar_label(ax, x, mu, sd, fmt="{:.2f}"):
    """Direct label placed clear of the error-bar cap (the contrast relief rule)."""
    if not np.isfinite(mu):
        return
    top = mu + (sd if np.isfinite(sd) else 0.0)
    ax.annotate(fmt.format(mu), (x, top), xytext=(0, 5), textcoords="offset points",
                ha="center", fontsize=7.5, color=INK)


def style():
    plt.rcParams.update({
        "figure.dpi": 130, "savefig.dpi": 300, "savefig.bbox": "tight",
        "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
        "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8,
        "axes.edgecolor": MUTED, "axes.linewidth": 0.8, "axes.labelcolor": INK,
        "text.color": INK, "xtick.color": INK2, "ytick.color": INK2,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
        "grid.alpha": 0.9, "axes.axisbelow": True,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False, "figure.facecolor": "white",
        "font.family": "DejaVu Sans", "mathtext.fontset": "dejavusans",
    })


def save(fig, out_dir, name):
    os.makedirs(out_dir, exist_ok=True)
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(out_dir, f"{name}.{ext}"))
    plt.close(fig)
    print(f"  wrote {name}.png/.pdf")


def load(results, name):
    p = os.path.join(results, f"{name}.json")
    if not os.path.exists(p):
        return None
    with open(p) as fh:
        return json.load(fh)


def _mean_std(agg, cond, metric):
    d = agg[cond][metric]
    return d.get("mean", np.nan), d.get("std", np.nan)


# --------------------------------------------------------------------------- #
# T1 -- the Born-rule flip model (closed form vs Monte-Carlo)
# --------------------------------------------------------------------------- #
def fig_flip_model(sanity, out):
    t1 = sanity.get("T1")
    if not t1:
        return
    m = np.array(t1["margins"])
    S = np.array(t1["shots"])
    closed = np.array(t1["p_flip_closed"])
    mc = np.array(t1["p_flip_montecarlo"])

    fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.1))
    ax = axes[0]
    cmap = plt.get_cmap(SEQ_CMAP)
    cols = cmap(np.linspace(0.35, 0.95, len(S)))          # sequential: shots = magnitude
    mm = np.linspace(1e-3, 0.99, 300)
    y_lab = 0.15                      # each curve meets this level at its own m ~ z/sqrt(S)
    for j, s in enumerate(S):
        ax.plot(mm, p_flip(mm, int(s)), color=cols[j], lw=1.6, zorder=2)
        ax.plot(m, mc[:, j], ls="none", marker="o", ms=3.4, mfc="white",
                mec=cols[j], mew=0.9, zorder=3)
        # direct-label along the curve so the labels spread instead of colliding at m->0
        m_lab = float(np.clip(1.036 / np.sqrt(float(s)), 0.012, 0.92))
        ax.annotate(f"S={s}", (m_lab, y_lab), xytext=(4, 6),
                    textcoords="offset points", color=cols[j], fontsize=7,
                    va="bottom", ha="left", rotation=0)
    ax.axhline(0.5, color=MUTED, lw=0.8, ls=":", zorder=1)
    ax.set_xlabel(r"Born margin  $m=|f_\theta(x)|$")
    ax.set_ylabel(r"flip probability  $p_{\mathrm{flip}}$")
    ax.set_xlim(-0.01, 0.8)
    ax.set_ylim(-0.02, 0.55)
    ax.set_title("Closed form (lines) vs Monte-Carlo (points)", color=INK)

    ax = axes[1]
    lim = max(closed.max(), mc.max()) * 1.05
    ax.plot([0, lim], [0, lim], color=MUTED, lw=0.9, ls="--", zorder=1)
    for j, s in enumerate(S):
        ax.plot(closed[:, j], mc[:, j], ls="none", marker=MARKERS[j % len(MARKERS)],
                ms=3.6, mfc="white", mec=cols[j], mew=0.9, label=f"S={s}", zorder=3)
    ax.set_xlabel(r"predicted $p_{\mathrm{flip}}$ (CLT model)")
    ax.set_ylabel(r"empirical $p_{\mathrm{flip}}$ (Binomial MC)")
    ax.set_title("Model vs measurement", color=INK)
    ax.legend(ncol=2, loc="lower right")
    fig.suptitle("T1  Born-rule flip model validated against the exact stochastic oracle",
                 y=1.03, fontsize=10.5, color=INK)
    save(fig, out, "fig_T1_flip_model")


# --------------------------------------------------------------------------- #
# T2 / T3 -- limits and ground truth
# --------------------------------------------------------------------------- #
def fig_limits(sanity, out):
    t2, t3 = sanity.get("T2"), sanity.get("T3")
    if not (t2 or t3):
        return
    fig, axes = plt.subplots(1, 3, figsize=(10.4, 3.1))

    ax = axes[0]
    if t2:
        sp = np.array(t2["probe_shots"], float)
        pert = np.array(t2["calibrated_median_perturbation"], float)
        det = t2["deterministic_hsja"]["median_perturbation"]
        ax.axhline(det, color=INK2, lw=1.2, ls="--", zorder=2)
        ax.annotate("deterministic HSJA  ($S\\to\\infty$)", (sp[0], det),
                    xytext=(2, 5), textcoords="offset points", fontsize=7.5, color=INK2)
        ax.plot(sp, pert, color=PALETTE[0], lw=1.8, marker=MARKERS[0], ms=5,
                mfc="white", mew=1.4, zorder=3)
        ax.set_xscale("log")
        ax.set_xlabel("probe shots  $S_{\\mathrm{probe}}$   (decision cap $\\propto S$)")
        ax.set_ylabel(r"median $\ell_2$ perturbation")
        ax.set_title("T2  median vs shot budget", color=INK, fontsize=9.5)

    ax = axes[1]
    gaps = (t2 or {}).get("paired_relative_gap")
    if gaps:
        # the paired per-image gap is the statistic that actually measures convergence:
        # the median above is taken over whichever images succeeded, so it also moves
        # with the success set.
        sp = np.array(t2["probe_shots"], float)
        ax.plot(sp, gaps, color=PALETTE[5], lw=1.8, marker=MARKERS[1], ms=5,
                mfc="white", mew=1.4, ls=LINESTYLES[1], zorder=3)
        ax.axhline(0.15, color=MUTED, lw=1.0, ls=":", zorder=2)
        ax.annotate("15% tolerance", (sp[0], 0.15), xytext=(2, 4),
                    textcoords="offset points", fontsize=7.5, color=INK2)
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel(r"probe shots  $S_{\mathrm{probe}}$")
        ax.set_ylabel("paired relative gap to deterministic")
        ax.set_title("T2  per-image convergence", color=INK, fontsize=9.5)

    ax = axes[2]
    if t3:
        rec = t3["per_point"]
        true = np.array([r["true_distance"] for r in rec])
        got = np.array([r["recovered"] for r in rec])
        lim = max(true.max(), got.max()) * 1.08
        ax.plot([0, lim], [0, lim], color=MUTED, lw=0.9, ls="--", zorder=1)
        ax.plot(true, got, ls="none", marker=MARKERS[1], ms=5.5, mfc="white",
                mec=PALETTE[1], mew=1.3, zorder=3)
        ax.set_xlabel("true point-to-boundary distance")
        ax.set_ylabel("perturbation recovered by attack")
        ax.set_title("T3  known analytic boundary", color=INK, fontsize=9.5)
        ax.set_xlim(0, lim)
        ax.set_ylim(0, lim)
    fig.tight_layout()
    save(fig, out, "fig_T2_T3_limits")


# --------------------------------------------------------------------------- #
# RQ1 -- feasibility across the attack suite
# --------------------------------------------------------------------------- #
def fig_rq1(rq1, out):
    if not rq1:
        return
    agg = rq1["aggregated"]
    order = [m for m in ["calibrated_hsja", "popskipjump", "fixed_hsja", "transfer",
                         "classical_hsja", "pgd_whitebox", "momentum"] if m in agg]
    fig, axes = plt.subplots(2, 1, figsize=(6.6, 5.4), sharex=True)

    ax = axes[0]
    for i, mth in enumerate(order):
        mu, sd = _mean_std(agg, mth, "success_rate")
        ax.bar(i, mu, yerr=sd, width=0.62, color=PALETTE[METHOD_SLOT[mth]],
               ecolor=INK2, capsize=3, error_kw={"lw": 1.0}, zorder=3)
        bar_label(ax, i, mu, sd)                      # direct label (relief rule)
    ax.set_ylabel("verified attack success rate")
    ax.set_ylim(0, 1.18)

    ax = axes[1]
    for i, mth in enumerate(order):
        mu, sd = _mean_std(agg, mth, "median_perturbation")
        ax.bar(i, mu, yerr=sd, width=0.62, color=PALETTE[METHOD_SLOT[mth]],
               ecolor=INK2, capsize=3, error_kw={"lw": 1.0}, zorder=3)
        bar_label(ax, i, mu, sd)
    ax.set_ylabel(r"median $\ell_2$ perturbation")
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels([METHOD_SHORT[m] for m in order], fontsize=7.5)
    fig.suptitle("RQ1  Hard-label attacks on a variational quantum classifier\n"
                 "(success is verified against the exact model; bars are mean $\\pm$ s.d. over seeds)",
                 y=1.0, fontsize=10, color=INK)
    save(fig, out, "fig_RQ1_feasibility")


# --------------------------------------------------------------------------- #
# RQ2 -- budget economics
# --------------------------------------------------------------------------- #
def fig_rq2(rq2, out):
    if not rq2:
        return
    # (a) budget curve
    bc = rq2.get("budget_curve")
    if bc:
        fig, ax = plt.subplots(figsize=(5.4, 3.4))
        agg = bc["aggregated"]
        xs = sorted(int(k) for k in agg)
        mu = [agg[str(x)]["median_perturbation"]["mean"] if str(x) in agg
              else agg[x]["median_perturbation"]["mean"] for x in xs]
        sd = [agg[str(x)]["median_perturbation"]["std"] if str(x) in agg
              else agg[x]["median_perturbation"]["std"] for x in xs]
        mu, sd = np.array(mu, float), np.array(sd, float)
        ax.plot(xs, mu, color=PALETTE[0], lw=1.8, marker=MARKERS[0], ms=5,
                mfc="white", mew=1.4, zorder=3)
        ax.fill_between(xs, mu - sd, mu + sd, color=PALETTE[0], alpha=0.16, lw=0, zorder=2)
        ax.set_xscale("log")
        ax.set_xlabel("total measurement budget  $T$  (shots)")
        ax.set_ylabel(r"median $\ell_2$ perturbation")
        ax.set_title("RQ2  Perturbation vs measurement budget", color=INK)
        save(fig, out, "fig_RQ2_budget_curve")

    # (b) allocation sweep + theory  -- two stacked panels, NEVER a dual axis
    al = rq2.get("allocation_sweep")
    if al:
        fig, axes = plt.subplots(2, 1, figsize=(5.6, 5.0), sharex=True)
        xs = np.array(al["probe_shots"], float)
        ys = np.array(al["median_perturbation"], float)
        agg = al["aggregated"]
        sd = np.array([agg[str(int(x))]["median_perturbation"]["std"]
                       if str(int(x)) in agg else np.nan for x in xs], float)
        ax = axes[0]
        ax.plot(xs, ys, color=PALETTE[0], lw=1.8, marker=MARKERS[0], ms=5,
                mfc="white", mew=1.4, zorder=3)
        ax.fill_between(xs, ys - sd, ys + sd, color=PALETTE[0], alpha=0.16, lw=0, zorder=2)
        io = al.get("interior_optimum", {})
        if io.get("argmin_x") is not None and np.isfinite(io.get("min_y", np.nan)):
            ax.plot([io["argmin_x"]], [io["min_y"]], marker="*", ms=13,
                    color=PALETTE[5], ls="none", zorder=4)
        ax.set_xscale("log")
        ax.set_ylabel(r"median $\ell_2$ perturbation")
        ax.set_title("RQ2  Shot allocation of the normal estimate at fixed $T$", color=INK)

        ax = axes[1]                       # the closed-form objective being optimised
        Sg = np.unique(np.round(np.geomspace(1, max(xs.max(), 2), 200)).astype(int))
        for j, mgn in enumerate([0.02, 0.05, 0.10, 0.20]):
            g = np.array([grad_snr_per_budget(mgn, int(s)) for s in Sg])
            ax.plot(Sg, g / g.max(), lw=1.5, color=PALETTE[[1, 3, 4, 6][j]],
                    ls=LINESTYLES[j], label=f"$m$={mgn:g}", zorder=3)
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel(r"probe shots  $S_{\mathrm{probe}}$   ($B=T_{\mathrm{grad}}/S$)")
        ax.set_ylabel("normalised  $(1-2p_{\\mathrm{flip}})^2/S$")
        ax.legend(ncol=4, loc="lower left")
        save(fig, out, "fig_RQ2_allocation")

    # (c) M1/M2 split
    sp = rq2.get("m1_m2_split")
    if sp:
        fig, ax = plt.subplots(figsize=(5.4, 3.4))
        xs = np.array(sp["grad_budget_frac"], float)
        ys = np.array(sp["median_perturbation"], float)
        ax.plot(xs, ys, color=PALETTE[0], lw=1.8, marker=MARKERS[0], ms=5,
                mfc="white", mew=1.4, zorder=3)
        io = sp.get("interior_optimum", {})
        if io.get("argmin_x") is not None and np.isfinite(io.get("min_y", np.nan)):
            ax.plot([io["argmin_x"]], [io["min_y"]], marker="*", ms=13,
                    color=PALETTE[5], ls="none", zorder=4)
        ax.set_xlabel("fraction of budget to normal estimation (M2)   $\\leftarrow$ M1 boundary search")
        ax.set_ylabel(r"median $\ell_2$ perturbation")
        ax.set_title("RQ2  Splitting the budget between boundary search and gradient",
                     color=INK)
        save(fig, out, "fig_RQ2_m1_m2_split")


# --------------------------------------------------------------------------- #
# RQ3 -- calibration payoff
# --------------------------------------------------------------------------- #
def fig_rq3(rq3, out):
    if not rq3:
        return
    agg = rq3["aggregated"]
    methods = rq3["methods"]
    budgets = rq3["budgets"]
    fig, axes = plt.subplots(2, 1, figsize=(5.8, 5.4), sharex=True)
    for i, mth in enumerate(methods):
        xs, su, sd_s, pe, sd_p = [], [], [], [], []
        for T in budgets:
            k = f"{mth}@T={T}"
            if k not in agg:
                continue
            xs.append(T)
            su.append(agg[k]["success_rate"]["mean"])
            sd_s.append(agg[k]["success_rate"]["std"])
            pe.append(agg[k]["median_perturbation"]["mean"])
            sd_p.append(agg[k]["median_perturbation"]["std"])
        c = PALETTE[METHOD_SLOT[mth]]
        axes[0].errorbar(xs, su, yerr=sd_s, color=c, lw=1.8, marker=MARKERS[i], ms=5,
                         mfc="white", mew=1.4, ls=LINESTYLES[i], capsize=2.5,
                         label=METHOD_LABEL[mth], zorder=3)
        axes[1].errorbar(xs, pe, yerr=sd_p, color=c, lw=1.8, marker=MARKERS[i], ms=5,
                         mfc="white", mew=1.4, ls=LINESTYLES[i], capsize=2.5, zorder=3)
    axes[0].set_ylabel("verified success rate")
    axes[0].set_ylim(-0.05, 1.1)
    axes[0].legend(loc="best")
    axes[1].set_ylabel(r"median $\ell_2$ perturbation")
    axes[1].set_xlabel("total measurement budget  $T$  (shots)")
    axes[1].set_xscale("log")
    fig.suptitle("RQ3  Born-rule calibration vs a constant-noise treatment at equal budget",
                 y=1.0, fontsize=10, color=INK)
    save(fig, out, "fig_RQ3_calibration")


# --------------------------------------------------------------------------- #
# RQ4 -- defenses vs a gradient-free attacker
# --------------------------------------------------------------------------- #
def fig_rq4(rq4, out):
    if not rq4:
        return
    agg = rq4["aggregated"]
    defs_ = ["none", "depolarizing", "randomized_encoding"]
    atks = ["calibrated_hsja", "pgd_whitebox"]

    def _key(d, a):
        """Resolve 'd|a', or the largest-n variant 'd|a|n=..' under an HLQ_RQ4_NS sweep."""
        if f"{d}|{a}" in agg:
            return f"{d}|{a}"
        cands = [k for k in agg if k.startswith(f"{d}|{a}|n=")]
        return max(cands, key=lambda s: int(s.split("n=")[1])) if cands else None

    if not any(_key(d, a) for d in defs_ for a in atks):
        return
    fig, axes = plt.subplots(2, 1, figsize=(6.4, 5.4), sharex=True)
    w = 0.36
    for ai, a in enumerate(atks):
        xs, mu, sd, cl = [], [], [], []
        for di, d in enumerate(defs_):
            k = _key(d, a)
            if k is None:
                continue
            xs.append(di + (ai - 0.5) * w)
            mu.append(agg[k]["median_perturbation"]["mean"])
            sd.append(agg[k]["median_perturbation"]["std"])
            cl.append(agg[k]["clean_accuracy"]["mean"])
        axes[0].bar(xs, mu, yerr=sd, width=w, color=PALETTE[METHOD_SLOT[a]],
                    ecolor=INK2, capsize=3, error_kw={"lw": 1.0},
                    label=METHOD_LABEL[a], zorder=3)
    axes[0].set_ylabel(r"median $\ell_2$ perturbation")
    axes[0].legend(loc="best")

    # the RQ5 guardrail travels with every robustness number
    for di, d in enumerate(defs_):
        k = f"{d}|calibrated_hsja"
        if k not in agg:
            continue
        acc = agg[k]["clean_accuracy"]["mean"]
        sd_acc = agg[k]["clean_accuracy"]["std"]
        axes[1].bar(di, acc, yerr=sd_acc, width=0.5,
                    color=PALETTE[2], ecolor=INK2, capsize=3, zorder=3)
        bar_label(axes[1], di, acc, sd_acc)
    axes[1].axhline(0.5, color=INK2, lw=1.1, ls="--", zorder=4)
    axes[1].annotate("chance", (len(defs_) - 0.55, 0.5), xytext=(0, 3),
                     textcoords="offset points", fontsize=7.5, color=INK2)
    axes[1].set_ylabel("clean accuracy")
    axes[1].set_ylim(0, 1.05)
    axes[1].set_xticks(range(len(defs_)))
    axes[1].set_xticklabels(["no defense", "depolarizing noise", "randomized encoding"])
    fig.suptitle("RQ4  Defenses validated against gradient attacks, re-tested gradient-free",
                 y=1.0, fontsize=10, color=INK)
    save(fig, out, "fig_RQ4_defenses")


# --------------------------------------------------------------------------- #
# RQ5 -- concentration vs robustness
# --------------------------------------------------------------------------- #
def fig_rq5(rq5, sanity, out):
    fits = (rq5 or {}).get("concentration_fits")
    t6 = (sanity or {}).get("T6")
    if not (fits or t6):
        return
    fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.3))
    fig.subplots_adjust(wspace=0.32)

    ax = axes[0]
    if fits:
        for i, (obs, d) in enumerate(fits.items()):
            ns = np.array(d["n_qubits"], float)
            v = np.array(d["var_f"], float)
            c = PALETTE[[0, 5][i % 2]]
            ax.plot(ns, v, color=c, lw=1.8, marker=MARKERS[i], ms=5.5, mfc="white",
                    mew=1.4, ls=LINESTYLES[i], zorder=3,
                    label=("local $Z_0$" if obs == "local_z" else r"global $Z^{\otimes n}$"))
            f = d["exponential_fit"]
            if np.isfinite(f.get("decay_rate_b", np.nan)):
                xx = np.linspace(ns.min(), ns.max(), 50)
                ax.plot(xx, np.exp(f["intercept"] - f["decay_rate_b"] * xx), color=c,
                        lw=0.9, ls=":", zorder=2)
    elif t6:
        ns = np.array([r["n_qubits"] for r in t6["random_deep_global_z"]], float)
        v = np.array(t6["var_f"], float)
        ax.plot(ns, v, color=PALETTE[5], lw=1.8, marker=MARKERS[1], ms=5.5,
                mfc="white", mew=1.4, label=r"random deep, global $Z^{\otimes n}$", zorder=3)
    ax.set_yscale("log")
    ax.set_xlabel("qubits  $n$")
    ax.set_ylabel(r"$\mathrm{Var}[f_\theta(x)]$")
    ax.set_title("Decision-value variance (concentration)", color=INK, fontsize=9.5)
    ax.legend(loc="best")

    ax = axes[1]
    agg = (rq5 or {}).get("aggregated")
    if agg:
        for i, obs in enumerate(rq5["observables"]):
            ns, pert, acc = [], [], []
            for n in rq5["n_qubits"]:
                k = f"{obs}|n={n}"
                if k not in agg:
                    continue
                ns.append(n)
                pert.append(agg[k]["median_perturbation"]["mean"])
                acc.append(agg[k]["clean_accuracy"]["mean"])
            c = PALETTE[[0, 5][i % 2]]
            # robustness is only meaningful above chance -> size marks by that margin
            sizes = 20 + 260 * np.clip(np.array(acc) - 0.5, 0, None)
            ax.plot(ns, pert, color=c, lw=1.5, ls=LINESTYLES[i], zorder=2,
                    label=("local $Z_0$" if obs == "local_z" else r"global $Z^{\otimes n}$"))
            ax.scatter(ns, pert, s=sizes, facecolor="white", edgecolor=c, linewidth=1.4,
                       marker=MARKERS[i], zorder=3)
        ax.set_xlabel("qubits  $n$")
        ax.set_ylabel(r"median $\ell_2$ perturbation")
        ax.set_title(r"Apparent robustness (mark size $\propto$ acc. above chance)",
                     color=INK, fontsize=9.5)
        ax.legend(loc="best")
    fig.suptitle("RQ5  Separating genuine robustness from exponential concentration",
                 y=1.06, fontsize=10.5, color=INK)
    save(fig, out, "fig_RQ5_concentration")


# --------------------------------------------------------------------------- #
# T5 -- budget monotonicity on a common success set
# --------------------------------------------------------------------------- #
def fig_t5(sanity, out):
    t5 = (sanity or {}).get("T5")
    if not t5 or "median_perturbation_common_set" not in t5:
        return
    fig, ax = plt.subplots(figsize=(5.2, 3.2))
    T = np.array(t5["budgets"], float)
    ax.plot(T, t5["median_perturbation_common_set"], color=PALETTE[0], lw=1.8,
            marker=MARKERS[0], ms=5, mfc="white", mew=1.4, ls=LINESTYLES[0],
            label=f"common success set (n={t5.get('n_common_images','?')})", zorder=3)
    ax.plot(T, t5["median_perturbation_all_successes"], color=PALETTE[3], lw=1.5,
            marker=MARKERS[1], ms=4.5, mfc="white", mew=1.2, ls=LINESTYLES[1],
            label="all successes (selection-biased)", zorder=2)
    ax.set_xscale("log")
    ax.set_xlabel("total measurement budget  $T$  (shots)")
    ax.set_ylabel(r"median $\ell_2$ perturbation")
    ax.set_title("T5  Budget monotonicity", color=INK)
    ax.legend(loc="best")
    save(fig, out, "fig_T5_budget_monotonicity")


# --------------------------------------------------------------------------- #
# Ablations
# --------------------------------------------------------------------------- #
def fig_ablation(res, out, key, name, title, labels=None):
    if not res:
        return
    agg = res["aggregated"]
    conds = list(agg)
    fig, axes = plt.subplots(2, 1, figsize=(5.6, 5.0), sharex=True)
    for i, c in enumerate(conds):
        mu, sd = _mean_std(agg, c, "median_perturbation")
        axes[0].bar(i, mu, yerr=sd, width=0.55, color=PALETTE[i % len(PALETTE)],
                    ecolor=INK2, capsize=3, zorder=3)
        bar_label(axes[0], i, mu, sd)
        mu2, sd2 = _mean_std(agg, c, "clean_accuracy")
        axes[1].bar(i, mu2, yerr=sd2, width=0.55, color=PALETTE[i % len(PALETTE)],
                    ecolor=INK2, capsize=3, zorder=3)
    axes[0].set_ylabel(r"median $\ell_2$ perturbation")
    axes[1].axhline(0.5, color=INK2, lw=1.0, ls="--", zorder=4)
    axes[1].set_ylabel("clean accuracy")
    axes[1].set_ylim(0, 1.05)
    axes[1].set_xticks(range(len(conds)))
    axes[1].set_xticklabels([str(labels[c] if labels and c in labels else c)
                             for c in conds])
    fig.suptitle(title, y=1.0, fontsize=10, color=INK)
    save(fig, out, name)


# --------------------------------------------------------------------------- #
def main():
    ap = argparse.ArgumentParser()
    root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    ap.add_argument("--results", default=os.path.join(root, "results"))
    ap.add_argument("--figures", default=os.path.join(root, "figures"))
    args = ap.parse_args()
    style()
    print(f"[figures] reading {args.results}")

    sanity = load(args.results, "sanity") or {}
    if sanity:
        fig_flip_model(sanity, args.figures)
        fig_limits(sanity, args.figures)
    fig_rq1(load(args.results, "rq1"), args.figures)
    fig_rq2(load(args.results, "rq2"), args.figures)
    fig_rq3(load(args.results, "rq3"), args.figures)
    fig_rq4(load(args.results, "rq4"), args.figures)
    fig_rq5(load(args.results, "rq5"), sanity, args.figures)
    fig_t5(sanity, args.figures)
    fig_ablation(load(args.results, "ablation_encoding"), args.figures,
                 "encoding", "fig_ablation_encoding",
                 "Ablation C  Encoding vs hard-label attackability")
    fig_ablation(load(args.results, "ablation_depth"), args.figures,
                 "n_layers", "fig_ablation_depth",
                 "Ablation  Ansatz depth vs hard-label attackability")
    fig_ablation(load(args.results, "ablation_dataset"), args.figures,
                 "dataset", "fig_ablation_dataset",
                 "Ablation  Dataset vs hard-label attackability")
    print(f"[figures] done -> {args.figures}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile experiments/significance.py
"""Significance tests for the head-to-head claims (plan Sec. 6).

Reads the per-image records already stored in results/*.json and reports, for the
load-bearing contrasts, a bootstrap CI on the headline statistic plus a hypothesis
test with an effect size -- so "calibrated beats X" is backed by a p-value, not just
non-overlapping means. Pure post-processing: no experiments are re-run.

Run:  python experiments/significance.py
"""
from __future__ import annotations

import os
import sys

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from hlq.analysis import load_json, save_json
from hlq.metrics import bootstrap_ci, compare_perturbations, proportion_test

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RESULTS = os.path.join(ROOT, "results")


def _pool(cells):
    """Pool per-image (perturbation, success) across a list of cells."""
    perts, succ = [], []
    for c in cells:
        for r in c.get("per_image", []):
            perts.append(r.get("verified_perturbation", float("inf")))
            succ.append(bool(r.get("verified_success", False)))
    return np.array(perts, float), np.array(succ, bool)


def _by(cells, keyfn):
    out = {}
    for c in cells:
        out.setdefault(keyfn(c), []).append(c)
    return out


def _contrast(name, cells_a, cells_b):
    pa, sa = _pool(cells_a)
    pb, sb = _pool(cells_b)
    return {
        "contrast": name,
        "success_rate": proportion_test(int(sa.sum()), len(sa), int(sb.sum()), len(sb)),
        "perturbation_mannwhitney": compare_perturbations(pa[sa], pb[sb]),
        "median_ci_a": bootstrap_ci(pa[sa]),
        "median_ci_b": bootstrap_ci(pb[sb]),
    }


def analyse():
    out = {}

    rq1 = load_json(os.path.join(RESULTS, "rq1.json"))
    by = _by(rq1["cells"], lambda c: c["attack"]["name"])
    # our method vs each hard-label baseline
    out["rq1"] = {c: _contrast(f"calibrated_vs_{c}", by["calibrated_hsja"], by[c])
                  for c in ("popskipjump", "fixed_hsja") if c in by}

    rq3 = load_json(os.path.join(RESULTS, "rq3.json"))
    by = _by(rq3["cells"], lambda c: (c["attack"]["name"], c["attack"]["total_budget"]))
    budgets = sorted({b for (_, b) in by})
    out["rq3"] = {}
    for T in budgets:
        ca = by.get(("calibrated_hsja", T), [])
        for base in ("popskipjump", "fixed_hsja"):
            cb = by.get((base, T), [])
            if ca and cb:
                out["rq3"][f"T={T}:calibrated_vs_{base}"] = _contrast(
                    f"calibrated_vs_{base}@T={T}", ca, cb)

    rq4 = load_json(os.path.join(RESULTS, "rq4.json"))
    # defense effect on the calibrated (gradient-free) attacker: none vs each defense
    def is_cal(c):
        return c["attack"]["name"] == "calibrated_hsja"
    by = _by([c for c in rq4["cells"] if is_cal(c)], lambda c: c["defense"]["name"])
    out["rq4"] = {}
    if "none" in by:
        for d in ("depolarizing", "randomized_encoding"):
            if d in by:
                out["rq4"][f"none_vs_{d}"] = _contrast(f"none_vs_{d}", by["none"], by[d])
                # is the defended clean accuracy significantly below chance?
                accs = [c["summary"]["clean_accuracy"] for c in by[d]]
                out["rq4"][f"{d}_clean_acc_ci"] = bootstrap_ci(accs, stat="mean")

    path = os.path.join(RESULTS, "significance.json")
    save_json(out, path)

    # human-readable digest
    print(f"[significance] wrote {path}\n")
    for rq, blk in out.items():
        print(f"=== {rq} ===")
        for name, res in blk.items():
            if "success_rate" in res:
                sr = res["success_rate"]
                mw = res["perturbation_mannwhitney"]
                print(f"  {name}: success {sr.get('prop_a', float('nan')):.2f} vs "
                      f"{sr.get('prop_b', float('nan')):.2f} (p={sr.get('p_value', float('nan')):.1e}); "
                      f"pert MWU p={mw.get('p_value', float('nan')):.1e} "
                      f"effect={mw.get('rank_biserial_effect', float('nan')):+.2f}")
            elif "point" in res:
                print(f"  {name}: {res['point']:.3f}  CI[{res['lo']:.3f}, {res['hi']:.3f}]")
    return out


if __name__ == "__main__":
    analyse()


## 3. Sanity gate (T1-T6)

Defined *before* the code, per the plan. **T1-T4 must pass before any attack number is trusted.**

* **T1** Born-rule flip model vs Monte-Carlo, and vs PennyLane's own shot sampling
* **T2** infinite-shot limit -> deterministic HopSkipJump (paired per-image)
* **T3** attack vs an analytically known boundary
* **T4** already-adversarial input returned unchanged
* **T5** budget monotonicity
* **T6** concentration guardrail fires on a concentrated model, not a trained one

In [ ]:
!python experiments/run_sanity.py --quick        # drop --quick for the full gate

In [ ]:
import json
s = json.load(open("results/sanity.json"))
for k, v in s["_summary"]["passed"].items():
    print(f"{k}: {'PASS' if v else 'FAIL'}")
print("\nT1-T4 trust gate:", "PASS" if s["_summary"]["trust_gate_T1_T4"] else "FAIL")


## 4. Experiments

Each research question varies exactly one axis with principled defaults for the rest.

| RQ | Question |
|---|---|
| RQ1 | Can a hard-label attack fool a VQC, and at what query cost? |
| RQ2 | How should a fixed measurement budget `T = Q x S` be allocated? |
| RQ3 | Does Born-rule calibration beat a constant-noise treatment (PopSkipJump)? |
| RQ4 | Do the quantum-noise / randomized-encoding defenses survive a *gradient-free* attacker? |
| RQ5 | Is apparent robustness really exponential concentration? |

`smoke` proves the pipeline; `medium` gives real trends; `full` is the heavier-statistics scope from the plan (250 images/cell, 8 seeds).

In [ ]:
# PUBLICATION-GRADE STATISTICS (8 seeds). Because the head-to-head blocks and the
# training-heavy RQ5 don't both fit one 12h CPU session, run this in TWO sessions:
#
#   RUN 1 (head-to-head, ~8-9h):  RQS = RUN1   (rq1/rq3/rq4/rq2 at 8 seeds)
#   RUN 2 (concentration, ~4h):   RQS = RUN2   (rq5, self-capped to 5 seeds)
#
# Each run is self-contained and downloads its own results zip (last cell). Merge both
# results/ folders locally and regenerate figures. Every cell is CHECKPOINTED, so a
# timeout inside a session loses nothing and re-running resumes.
PRESET = "kaggle8"         # 8 seeds. Use "kaggle" (3 seeds) for a faster first pass.
JOBS   = 4                 # parallel cells; set to the session's core count
RUN1   = ["rq1", "rq3", "rq4", "rq2"]     # rq2 last: its allocation sweep is the slowest
RUN2   = ["rq5"]
RUN3   = ["rq2"]           # finish RQ2 only (see HLQ_RQ2_PARTS below)
RQS    = RUN1              # <- switch to RUN2 / RUN3 for later sessions

# RQ2 splits into three independent sweeps, so an interrupted run can be finished
# without redoing the completed parts. For RUN3, select just what is missing, e.g.:
#   import os; os.environ["HLQ_RQ2_PARTS"] = "m1m2"        # the interior-optimum sweep
#   import os; os.environ["HLQ_RQ2_PARTS"] = "m1m2,alloc"  # both remaining sweeps

# RQ4 across qubit count (tests Theorem 4's exponential scaling): os.environ["HLQ_RQ4_NS"]="4,6"
# RQ5 n=12 tier (tens of min/model training):                     os.environ["HLQ_RQ5_MAX_N"]="12"
ALL_RQS = ["rq1", "rq2", "rq3", "rq4", "rq5", "ablation_encoding",
           "ablation_depth", "ablation_dataset"]

import subprocess, sys
for rq in RQS:
    print(f"\n########## {rq} ##########", flush=True)
    subprocess.run([sys.executable, "experiments/driver.py", "--rq", rq,
                    "--preset", PRESET, "--jobs", str(JOBS)], check=False)


## 5. Figures

Research figures rendered from the results JSON: a validated colorblind-safe categorical palette in fixed order, always with a second encoding (marker + linestyle); single-hue sequential ramps for magnitude; no dual axes anywhere.

In [ ]:
!python experiments/make_figures.py

In [ ]:
from IPython.display import Image, display
import glob, os
for p in sorted(glob.glob("figures/*.png")):
    print("\n" + os.path.basename(p))
    display(Image(p))


## 5b. Significance tests

Bootstrap CIs + Mann-Whitney / two-proportion tests on the per-image data, so each head-to-head claim carries a p-value and an effect size (plan Sec. 6). Runs only for whichever RQs are present.

In [ ]:
!python experiments/significance.py

## 6. Results summary

The headline numbers, straight out of the JSON.

In [ ]:
import json, os, numpy as np

def show(rq):
    p = f"results/{rq}.json"
    if not os.path.exists(p):
        return
    d = json.load(open(p))
    agg = d.get("aggregated")
    if not agg:
        return
    print(f"\n=== {rq} ===")
    for cond, mets in agg.items():
        sr = mets.get("success_rate", {})
        mp = mets.get("median_perturbation", {})
        print(f"  {str(cond):42s} success={sr.get('mean', float('nan')):.3f}"
              f" +-{sr.get('std', float('nan')):.3f}   median_pert="
              f"{mp.get('mean', float('nan')):.4f} +-{mp.get('std', float('nan')):.4f}")

for rq in ["rq1", "rq3", "rq4", "rq5", "ablation_encoding", "ablation_depth",
           "ablation_dataset"]:
    show(rq)

if os.path.exists("results/rq2.json"):
    d = json.load(open("results/rq2.json"))
    print("\n=== rq2: allocation (H2 interior optimum?) ===")
    print(" ", d["allocation_sweep"]["interior_optimum"])
    print("=== rq2: M1/M2 split ===")
    print(" ", d["m1_m2_split"]["interior_optimum"])


## 7. Collect outputs

All results are JSON; figures are PNG + vector PDF.

In [ ]:
import shutil, os
shutil.make_archive("hlq_results", "zip", ".", "results")
shutil.make_archive("hlq_figures", "zip", ".", "figures")
print("wrote hlq_results.zip and hlq_figures.zip")
print("results:", sorted(os.listdir("results")))
print("figures:", sorted(os.listdir("figures")))
